## Setup & Data Loading
Load transaction and auction CSVs, parse datetimes, and filter to the analysis period.

In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats

# Create figures directory if it doesn't exist
import os

if not os.path.exists("figures"):
    os.makedirs("figures")

# Read all CSV files from the specified directory
csv_files = glob.glob(f"data/case_study/searcher_txs_with_pnl_merged.csv")
txs = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
auctions = pd.read_csv("data/case_study/timeboost_auctions_casestudy.csv")

# Drop any auction-derived columns that may have been added in a previous run
auction_cols = ['merged', 'auction_round', 'winner_address', 'winner_name',
                'top_bid_eth', 'paid_bid_eth', 'express_lane_controller_address',
                'round_start_time', 'round_end_time', 'top_bid_usd', 'paid_bid_usd']
txs = txs.drop(columns=[c for c in auction_cols if c in txs.columns])

txs["timeboosted"] = txs["timeboosted"].fillna(False)

txs["block_time"] = pd.to_datetime(txs["block_time"], utc=True).dt.tz_convert(None)
auctions["round_start_time"] = pd.to_datetime(auctions["round_start_time"], utc=True)
auctions["round_end_time"]   = pd.to_datetime(auctions["round_end_time"], utc=True)

# If winner_name is Unlabeled, set it to Unknown
auctions["winner_name"] = auctions["winner_name"].replace("Unlabeled", "Unknown")

txs = txs.sort_values("block_time")
auctions = auctions.sort_values("round_start_time")

In [ ]:
# Print timeboosted and non-timeboosted transaction counts per contract
print("Timeboosted and non-timeboosted transaction counts per contract:")
print(txs.groupby(["tx_to_address", "timeboosted"]).size().unstack(fill_value=0))

# Total tb txs: 5850610 (not only cexdex)
# Total sel tb txs: 601286 (not only cexdex)
# Identified sel cexdex tb txs: 418622
# Identified sel cexdex non-tb txs: 298124
# Total wm tb txs: 4515547 (not only cexdex)
# Identified wm cexdex tb txs: 2345100
# Identified wm cexdex non-tb txs: 1289841  

In [ ]:

FIG = {
    "wide":       (14, 5),   # standard time-series
    "wide_short": (14, 4),   # compact time-series
    "wide_tall":  (14, 8),   # two-panel stacked
    "double":     (16, 5),   # side-by-side panels
    "square":     (8,  5),   # single-panel
}

FONT = {
    "title":  16,
    "label":  15,
    "tick":   15,
    "legend": 13,
    "annot":  13,
}

import matplotlib as mpl
mpl.rcParams.update({
    "figure.figsize":        FIG["wide"],
    "axes.titlesize":        FONT["title"],
    "axes.labelsize":        FONT["label"],
    "xtick.labelsize":       FONT["tick"],
    "ytick.labelsize":       FONT["tick"],
    "legend.fontsize":       FONT["legend"],
    "axes.grid.axis":        "y",
    "grid.linestyle":        "--",
    "grid.alpha":            0.4,
    "figure.autolayout":     True,
})


In [ ]:
ZONE_LABELS = [
    (None,                                           pd.Timestamp("2026-02-12 20:30:51", tz="UTC"), "Pre-Kairos"),
    (pd.Timestamp("2026-02-12 20:30:51", tz="UTC"), pd.Timestamp("2026-02-18 20:01:52", tz="UTC"), "Kairos"),
    (pd.Timestamp("2026-02-18 20:01:52", tz="UTC"), pd.Timestamp("2026-02-24 01:16:50", tz="UTC"), "Reserve Price\nAdaptation"),
    (pd.Timestamp("2026-02-24 01:16:50", tz="UTC"), None,                                          "Steady State"),
]

ZONE_COLORS = ["#a8d4ed", "#fbc88a", "#a8d9a8", "#f0b8d0"]

# Reserve price per zone (ETH)
RESERVE_PRICE_ETH = {
    "Pre-Kairos":               0.001,
    "Kairos":                   0.001,
    "Reserve Price\nAdaptation":0.0075,
    "Steady State":             0.001,
}

def _assign_zone(d):
    """Assign a Python date to a zone using UTC-precise ZONE_LABELS boundaries."""
    ts = pd.Timestamp(d, tz="UTC")
    for z_start, z_end, label in ZONE_LABELS:
        if (z_start is None or ts >= z_start) and (z_end is None or ts < z_end):
            return label
    return "Unknown"

def _add_reserve_price_line(ax, t_min, t_max):
    """Overlay reserve price as a step line on a tz-naive datetime x-axis (ETH units)."""
    labeled = False
    for z_start, z_end, z_label in ZONE_LABELS:
        x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
        x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
        rp = RESERVE_PRICE_ETH[z_label]
        ax.hlines(rp, x0, x1,
                  colors="crimson", linewidth=1.8, linestyle="--", zorder=5,
                  label="Reserve Price" if not labeled else "_nolegend_")
        labeled = True

# (timestamp, legend label, line color)
EVENT_LINES = [
    (pd.Timestamp("2026-02-12 20:30:51", tz="UTC"), "Kairos begins",    "#e6550d"),
    (pd.Timestamp("2026-02-18 20:01:52", tz="UTC"), "Reserve price â†‘", "#31a354"),
    (pd.Timestamp("2026-02-24 01:16:50", tz="UTC"), "Reserve price â†“", "#756bb1"),
]

def _add_event_lines(ax):
    """Draw vertical dashed lines at key event timestamps on a datetime x-axis."""
    for ts, label, color in EVENT_LINES:
        ax.axvline(ts.tz_convert(None).to_pydatetime(), color=color,
                   linewidth=2.0, linestyle="--", zorder=7)

## Price Feeds & Volatility
Load Binance 1-second price data and compute rolling ETH volatility.

In [ ]:
binance_btcusd = pd.read_csv(f"data/case_study/prices/BTCUSDT-1s-merged.csv")
binance_ethusd = pd.read_csv(f"data/case_study/prices/ETHUSDT-1s-merged.csv")
binance_arbusd = pd.read_csv(f"data/case_study/prices/ARBUSDT-1s-merged.csv")

cols = [
    "open_time_us", "open", "high", "low", "close", "volume",
    "close_time_us", "quote_vol", "trades", "taker_base", "taker_quote", "ignore"
]

binance_arbusd.columns = cols
binance_ethusd.columns = cols
binance_btcusd.columns = cols

HORIZONS = [0, 5, 10, 300, 600, 1800]

def prepare_pricefeed(df, token):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["open_time_us"] / 1e6, unit="s", utc=True)
    df["midprice"] = (df["high"] + df["low"]) / 2
    df = df[["timestamp", "midprice"]].sort_values("timestamp").reset_index(drop=True)
    df = df.rename(columns={"midprice": f"{token}_mid"})
    return df

binance_arbusd = prepare_pricefeed(binance_arbusd, "ARB")
binance_ethusd = prepare_pricefeed(binance_ethusd, "ETH")
binance_btcusd = prepare_pricefeed(binance_btcusd, "BTC")

pricefeeds = {
    "ARB": binance_arbusd.sort_values("timestamp").reset_index(drop=True),
    "ETH": binance_ethusd.sort_values("timestamp").reset_index(drop=True),
    "BTC": binance_btcusd.sort_values("timestamp").reset_index(drop=True),
}

In [ ]:
eth = binance_ethusd.sort_values("timestamp").copy()
eth["timestamp"] = pd.to_datetime(eth["timestamp"], utc=True)
eth["log_ret"] = np.log(eth["ETH_mid"] / eth["ETH_mid"].shift(1))

# 30-min rolling window (1800 seconds)
eth["vol_30m"] = eth["log_ret"].rolling(1800).std()

daily_vol = eth.groupby(eth["timestamp"].dt.date)["log_ret"].std()

daily_vol = daily_vol.reset_index().rename(columns={"log_ret": "daily_vol"})

daily_vol["date"] = pd.to_datetime(daily_vol["timestamp"], utc=True).dt.date

# # Daily volatility = mean of intraday rolling vol
# daily_vol = (
#     eth.assign(date=eth["timestamp"].dt.date)
#        .groupby("date")["vol_30m"]
#        .mean()
#        .reset_index()
#        .rename(columns={"vol_30m": "daily_vol"})
# )

## Data Enrichment
Merge ETH price into auctions, compute USD bid amounts, and join auction info to transactions.

In [ ]:
# Calculate USD for top_bid_eth and paid_bid_eth, using the eth price at auction start time
auctions = auctions.merge(
    binance_ethusd.rename(columns={"ETH_mid": "eth_price_at_auction_start"}),
    left_on="round_start_time",
    right_on="timestamp",
    how="left"
)
auctions["top_bid_usd"]  = auctions["top_bid_eth"]  * auctions["eth_price_at_auction_start"]
auctions["paid_bid_usd"] = auctions["paid_bid_eth"] * auctions["eth_price_at_auction_start"]

# Drop existing auction columns from txs if present
for col in ["round_start_time", "round_end_time", "auction_round", "top_bid_usd", "paid_bid_usd"]:
    if col in txs.columns:
        txs = txs.drop(columns=[col])

# Update txs with auction info using range join on block_time within [round_start_time, round_end_time]
auctions_tz_naive = auctions.copy()
auctions_tz_naive["round_start_time"] = auctions_tz_naive["round_start_time"].dt.tz_convert(None)
auctions_tz_naive["round_end_time"]   = auctions_tz_naive["round_end_time"].dt.tz_convert(None)

txs_enriched = pd.merge_asof(
    txs.sort_values("block_time"),
    auctions_tz_naive[["round_start_time", "round_end_time", "auction_round", "top_bid_usd", "paid_bid_usd", "winner_name"]].sort_values("round_start_time"),
    left_on="block_time",
    right_on="round_start_time",
    direction="backward"
)
# Nullify rows where block_time falls outside the matched round's window
mask = (
    txs_enriched["block_time"] >= txs_enriched["round_start_time"]
) & (
    txs_enriched["block_time"] <= txs_enriched["round_end_time"]
)
txs_enriched.loc[~mask, ["auction_round", "top_bid_usd", "paid_bid_usd", "round_start_time", "round_end_time"]] = None
txs = txs_enriched

txs["time_since_round_start"] = (txs["block_time"] - txs["round_start_time"]).dt.total_seconds()
txs["time_since_minute_start"] = (txs["block_time"] - txs["block_time"].dt.floor("T")).dt.total_seconds()

In [ ]:
# ETH price at block_time via merge_asof (last price at or before block_time)
# binance_ethusd timestamps are tz-aware UTC; block_time is tz-naive UTC - strip tz for alignment
eth_prices = binance_ethusd[["timestamp", "ETH_mid"]].copy()
eth_prices["timestamp"] = eth_prices["timestamp"].dt.tz_convert(None)

if "tx_fee_usd" in txs.columns:
    txs = txs.drop(columns=["tx_fee_usd", "eth_price_at_tx"], errors="ignore")

txs = pd.merge_asof(
    txs.sort_values("block_time"),
    eth_prices.rename(columns={"ETH_mid": "eth_price_at_tx"}).sort_values("timestamp"),
    left_on="block_time",
    right_on="timestamp",
    direction="backward",
)
txs["tx_fee_usd"] = txs["tx_fee_eth"] * txs["eth_price_at_tx"]


In [ ]:
SEARCHERS = {
    "Wintermute": [
        '0x27920e8039d2b6e93e36f5d5f53b998e2e631a70'
    ],
    "Selini": [
        '0xee2e7bbb67676292af2e31dffd1fea2276d6c7ba'
    ]
}
ALL_SEARCHER_ADDRS = [a for addrs in SEARCHERS.values() for a in addrs]

wm_txs = txs[txs["tx_to_address"].isin(SEARCHERS["Wintermute"])].copy()
sel_txs = txs[txs["tx_to_address"].isin(SEARCHERS["Selini"])].copy()

wm_txs["date"] = wm_txs["block_time"].dt.date
sel_txs["date"] = sel_txs["block_time"].dt.date

## Auction Winners

In [ ]:
print(len(auctions))
print(auctions["winner_name"].value_counts())
print(auctions["auction_round"].nunique())

print(auctions["auction_round"].value_counts().gt(1).sum(), "rounds with duplicates")
print(len(auctions), "total rows vs", auctions["auction_round"].nunique(), "unique rounds")


In [ ]:
WINNER_COLORS = {
    "Wintermute": "#1f77b4",
    "Selini":     "#ff7f0e",
    "Kairos":     "#9467bd",
    "Others":    "#7f7f7f",
}

# Daily winner counts, reindexed to full date range so gaps show as zero
auctions["date"] = auctions["round_start_time"].dt.tz_convert(None).dt.date
daily_winners = (
    auctions.groupby(["date", "winner_name"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=list(WINNER_COLORS.keys()), fill_value=0)
)

full_index = pd.date_range(
    start=daily_winners.index.min(),
    end=daily_winners.index.max(),
    freq="D",
).date
daily_winners = daily_winners.reindex(full_index, fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))

daily_winners.plot(
    kind="bar",
    ax=ax,
    color=[WINNER_COLORS[c] for c in daily_winners.columns],
    width=0.8,
    alpha=0.9,
    zorder=3,
)

# Zone shading
dates = list(daily_winners.index)
n = len(dates)

def date_to_x(ts):
    d = ts.tz_convert(None).date()
    for i, bar_date in enumerate(dates):
        if bar_date >= d:
            return i - 0.5
    return n - 0.5

for i, (z_start, z_end, z_label) in enumerate(ZONE_LABELS):
    x0 = -0.5      if z_start is None else date_to_x(z_start)
    x1 = n - 0.5   if z_end   is None else date_to_x(z_end)
    ax.axvspan(x0, x1, alpha=0.18, color=ZONE_COLORS[i], zorder=0)
    ax.text((x0 + x1) / 2, 1.01,
            z_label, ha="center", va="bottom", fontsize=10,
            color="0.3", style="italic", transform=ax.get_xaxis_transform(), clip_on=False)

ax.set_title("Daily Auction Round Winners Over Time")
ax.set_xlabel("")
ax.set_ylabel("Number of rounds won")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
ax.legend(title="Winner", loc="upper left", fontsize=FONT["legend"], title_fontsize=(FONT["legend"]+1))
ax.grid(axis="y", linestyle="--", alpha=0.4)
fig.tight_layout()
plt.savefig("figures/auction_winners_over_time.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:

# Stacked area — full period % share (top) + Feb 2026 absolute round counts (bottom)
# Every day = 1440 rounds; shortfall shown as hatched "No Auction" band
_auctions_full = pd.read_csv("data/case_study/timeboost_auctions.csv", parse_dates=["round_start_time"])
_auctions_full["round_start_time"] = pd.to_datetime(
    _auctions_full["round_start_time"].astype(str).str.replace(" UTC", ""),
    utc=True
)
_auctions_full["winner_name"] = _auctions_full["winner_name"].replace("Unlabeled", "Others")
_auctions_full["date"] = _auctions_full["round_start_time"].dt.tz_convert(None).dt.date

ROUNDS_PER_DAY = 1440
winners = list(WINNER_COLORS.keys())
colors  = [WINNER_COLORS[w] for w in winners]

_dw = (
    _auctions_full.groupby(["date", "winner_name"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=winners, fill_value=0)
)
_full_idx = pd.date_range(_dw.index.min(), _dw.index.max(), freq="D").date
_dw = _dw.reindex(_full_idx, fill_value=0)

x      = pd.DatetimeIndex([pd.Timestamp(d) for d in _dw.index])
pct    = _dw[winners].div(ROUNDS_PER_DAY) * 100

inset_start = pd.Timestamp("2026-02-01")
mask  = x >= inset_start
x_feb = x[mask]

import matplotlib.dates as mdates

def _draw(ax, xdata, df, cap):
    ax.stackplot(xdata, [df[w].values for w in winners], labels=winners, colors=colors, alpha=0.8, edgecolor="0.5", linewidth=1, zorder=4)
    bot = df[winners].sum(axis=1).values
    ax.fill_between(xdata, bot, cap,
                    facecolor="white", edgecolor="0.5", hatch="////",
                    linewidth=0, zorder=3, label="No Bidder")
    ax.set_xlim(xdata[0], xdata[-1])
    ax.set_ylim(0, cap)
    ax.yaxis.grid(True, linestyle="--", alpha=0.9, zorder=16)
    ax.set_axisbelow(True)

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=FIG["wide_tall"])

_draw(ax_top, x,     pct,          cap=100)
_draw(ax_bot, x_feb, _dw.loc[mask], cap=ROUNDS_PER_DAY)

# Top: monthly ticks — spans ~10 months
ax_top.xaxis.set_major_locator(mdates.MonthLocator())
ax_top.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))

# Bottom: weekly ticks — spans ~5 weeks
ax_bot.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax_bot.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

ax_top.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax_top.set_title("Timeboost Market Shares — All Time")
ax_top.set_ylabel("Share of rounds won")
handles, labels_ = ax_top.get_legend_handles_labels()
ax_top.legend(handles, labels_, title="Winner", loc="upper left",
              fontsize=FONT["legend"], title_fontsize=FONT["legend"] + 1)

ax_bot.set_title("Since Feb 2026 — Rounds Won per Day")
ax_bot.set_ylabel("Rounds won")

for ax in (ax_top, ax_bot):
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.tight_layout()
plt.savefig("figures/auction_winners_area_full.pdf", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

auctions_sorted = auctions.sort_values("round_start_time").copy()
ts = auctions_sorted["round_start_time"].dt.tz_convert(None)

ax1.plot(ts, auctions_sorted["top_bid_eth"],
         label="Top Bid", color="#1f77b4", linewidth=0.5, alpha=0.8)
ax1.plot(ts, auctions_sorted["paid_bid_eth"],
         label="Paid Bid", color="#ff7f0e", linewidth=0.5, alpha=0.8)
ax1.set_ylabel("Bid (ETH)")
ax1.grid(axis="y", linestyle="--", alpha=0.4)

t_min, t_max = ts.min(), ts.max()
ax1.set_xlim(t_min, t_max)

# _add_reserve_price_line(ax1, t_min, t_max)

# # Volatility on right axis
# ax2 = ax1.twinx()
# vol_ts = pd.to_datetime(daily_vol["date"])
# ax2.plot(vol_ts, daily_vol["daily_vol"],
#          color="#2ca02c", linewidth=2, alpha=0.85, label="ETH Daily Vol")
# ax2.set_ylabel("ETH Daily Volatility", color="#2ca02c", fontsize=FONT["label"])
# ax2.tick_params(axis="y", labelcolor="#2ca02c")
# ax2.yaxis.set_label_position("right")

# Zone shading
for i, (z_start, z_end, z_label) in enumerate(ZONE_LABELS):
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    ax1.axvspan(x0, x1, alpha=0.18, color=ZONE_COLORS[i], zorder=0)
    mid = x0 + (x1 - x0) / 2
    ax1.text(mid, 1.01, z_label, ha="center", va="bottom",
             fontsize=9, color="0.3", style="italic", transform=ax1.get_xaxis_transform(), clip_on=False)
_add_event_lines(ax1)

lines1, labs1 = ax1.get_legend_handles_labels()
# lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1, labs1, fontsize=FONT["legend"], loc="upper left")

# ax1.set_xlabel("Date")
ax1.set_title("Top Bid vs Paid Bid per Auction Round")
plt.setp(ax1.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig("figures/bids_per_round.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:

# Top Bid vs Paid Bid + Normalized Bid Gap — two panels, shared x-axis
import glob

_bid_files = sorted(glob.glob("data/case_study/bids/bids_*.csv"))
_bids = pd.concat([pd.read_csv(f) for f in _bid_files], ignore_index=True)

_round_times = auctions[["auction_round", "round_start_time"]].copy()
_rst = pd.to_datetime(_round_times["round_start_time"])
_round_times["round_start_time"] = _rst.dt.tz_convert(None) if _rst.dt.tz is not None else _rst

def _first_second(amounts):
    s = sorted(amounts, reverse=True)
    if len(s) < 2:
        return pd.Series({"gap_norm": float("nan")})
    return pd.Series({"gap_norm": (s[0] - s[1]) / s[0] if s[0] > 0 else float("nan")})

_gap = (
    _bids.groupby("Round")["Amount"]
    .apply(_first_second)
    .reset_index()
)
_gap.columns = ["Round", "metric", "value"]
_gap = _gap.pivot(index="Round", columns="metric", values="value").reset_index()
_gap = _gap.merge(_round_times, left_on="Round", right_on="auction_round", how="left")
_gap = _gap.dropna(subset=["round_start_time", "gap_norm"]).sort_values("round_start_time")

_daily_gap = (
    _gap.assign(date=lambda d: d["round_start_time"].dt.normalize())
    .groupby("date")["gap_norm"].median()
    .reset_index(name="median_gap_norm")
)
_t_max_gap = _gap["round_start_time"].max()
if _daily_gap["date"].max() < _t_max_gap:
    _daily_gap = pd.concat([
        _daily_gap,
        pd.DataFrame({"date": [_t_max_gap], "median_gap_norm": [_daily_gap["median_gap_norm"].iloc[-1]]})
    ], ignore_index=True)

_vol_full = (
    eth.groupby(eth["timestamp"].dt.date)["log_ret"]
    .std()
    .reset_index()
    .rename(columns={"timestamp": "date", "log_ret": "daily_vol"})
)
_vol_full["date"] = pd.to_datetime(_vol_full["date"])
# Extend to last auction date
_as_tmp = auctions.sort_values("round_start_time")
_t_max_vol = pd.Timestamp(_as_tmp["round_start_time"].iloc[-1]).tz_convert(None)
if _vol_full["date"].max() < _t_max_vol:
    _vol_full = pd.concat([
        _vol_full,
        pd.DataFrame({"date": [_t_max_vol], "daily_vol": [_vol_full["daily_vol"].iloc[-1]]})
    ], ignore_index=True)

_as = auctions.sort_values("round_start_time").copy()
ts   = _as["round_start_time"].dt.tz_convert(None)
t_min, t_max = ts.min(), ts.max()

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=FIG["wide_tall"], sharex=True)

for ax in (ax_top, ax_bot):
    for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
        x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
        x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
        ax.axvspan(x0, x1, alpha=0.12, color=ZONE_COLORS[i], zorder=0)

# Top: bids (left axis)
ax_top.plot(ts, _as["top_bid_eth"],  label="Top Bid",  color="#1f77b4", linewidth=1.5, alpha=0.8)
ax_top.plot(ts, _as["paid_bid_eth"], label="Paid Bid", color="#ff7f0e", linewidth=1.5, alpha=0.8)
_add_event_lines(ax_top)
ax_top.set_ylabel("Bid (ETH)")
ax_top.set_ylim(bottom=0)
ax_top.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0)
ax_top.set_axisbelow(True)

# Volatility on right axis — full date range
VOL_COLOR = "gray"
ax_vol = ax_top.twinx()
ax_vol.plot(_vol_full["date"], _vol_full["daily_vol"],
            color=VOL_COLOR, linewidth=2.2, linestyle=":", label="Volatility")
ax_vol.set_ylabel("ETH Daily Volatility", color=VOL_COLOR, fontsize=FONT["label"])
ax_vol.tick_params(axis="y", labelcolor=VOL_COLOR)

lines1, labs1 = ax_top.get_legend_handles_labels()
lines2, labs2 = ax_vol.get_legend_handles_labels()
ax_top.legend(lines1 + lines2, labs1 + labs2, fontsize=FONT["legend"], loc="upper right", bbox_to_anchor=(1, 0.92))

for z_start, z_end, z_label in ZONE_LABELS:
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax_top.text(mid, 1.01, z_label, ha="center", va="bottom",
                fontsize=FONT["annot"], color="0.25", style="italic", zorder=6, transform=ax_top.get_xaxis_transform(), clip_on=False)

# Bottom: normalized gap
ax_bot.scatter(_gap["round_start_time"], _gap["gap_norm"],
               s=2.5, alpha=0.25, color="#1f77b4", linewidths=0, label="Per-round")
ax_bot.plot(_daily_gap["date"], _daily_gap["median_gap_norm"],
            color="black", linewidth=1.8, label="Daily median", zorder=5, alpha=0.85)
_add_event_lines(ax_bot)
ax_bot.set_ylabel("Relative Bid Gap")
ax_bot.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax_bot.set_ylim(0, 1)
ax_bot.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0)
ax_bot.set_axisbelow(True)
ax_bot.legend(fontsize=FONT["legend"], loc="upper left")

ax_bot.set_xlim(t_min, t_max)
ax_bot.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax_bot.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.setp(ax_bot.get_xticklabels(), rotation=30, ha="right")
fig.suptitle("Auction Bids and Relative Bid Gap per Round", fontsize=FONT["title"])
fig.tight_layout()
plt.savefig("figures/bids_and_gap_stacked.pdf", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
# Daily median relative bid gap by zone
_gap_z = _gap.copy()
_gap_z["zone"] = _gap_z["round_start_time"].apply(lambda t: _assign_zone(t.date()))

print(f"{'Zone':<30} {'Median Gap':>12} {'N rounds':>10}")
print("-" * 54)
for _, _, z_label in ZONE_LABELS:
    z_data = _gap_z[_gap_z["zone"] == z_label]["gap_norm"].dropna()
    med = z_data.median()
    n   = len(z_data)
    print(f"{z_label.replace(chr(10), ' '):<30} {med:>11.1%} {n:>10,}")


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

auctions_sorted = auctions.sort_values("round_start_time").copy()
auctions_sorted["bid_gap_eth"] = auctions_sorted["top_bid_eth"] - auctions_sorted["paid_bid_eth"]
ts = auctions_sorted["round_start_time"].dt.tz_convert(None)

ax1.plot(ts, auctions_sorted["bid_gap_eth"],
         color="#1f77b4", linewidth=0.5, alpha=0.8, label="Bid Gap (ETH)")
ax1.set_ylabel("Bid Gap (ETH)")
ax1.grid(axis="y", linestyle="--", alpha=0.4)

t_min, t_max = ts.min(), ts.max()

_add_reserve_price_line(ax1, t_min, t_max)

# Volatility on right axis
ax2 = ax1.twinx()
vol_ts = pd.to_datetime(daily_vol["date"])
ax2.plot(vol_ts, daily_vol["daily_vol"],
         color="#2ca02c", linewidth=2, alpha=0.85, label="ETH Daily Vol")
ax2.set_ylabel("ETH Daily Volatility", color="#2ca02c", fontsize=FONT["label"])
ax2.tick_params(axis="y", labelcolor="#2ca02c")
ax2.yaxis.set_label_position("right")

# Zone shading
for i, (z_start, z_end, z_label) in enumerate(ZONE_LABELS):
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    ax1.axvspan(x0, x1, alpha=0.18, color=ZONE_COLORS[i], zorder=0)
    mid = x0 + (x1 - x0) / 2
    ax1.text(mid, 1.01, z_label, ha="center", va="bottom",
             fontsize=9, color="0.3", style="italic", transform=ax1.get_xaxis_transform(), clip_on=False)
_add_event_lines(ax1)

lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, fontsize=FONT["legend"], loc="upper left")

ax1.set_xlabel("Date")
ax1.set_title("Bid Gap per Auction Round vs ETH Volatility")
plt.setp(ax1.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig("figures/bid_gap_per_round.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
wm_addrs  = set(SEARCHERS["Wintermute"])
sel_addrs = set(SEARCHERS["Selini"])

tb_txs = txs[txs["timeboosted"] == True].copy()
tb_txs["is_wm"]  = tb_txs["tx_to_address"].isin(wm_addrs)
tb_txs["is_sel"] = tb_txs["tx_to_address"].isin(sel_addrs)

round_presence = (
    tb_txs.groupby("auction_round")
    .agg(has_wm=("is_wm", "any"), has_sel=("is_sel", "any"))
    .reset_index()
)

aug = auctions.merge(round_presence, on="auction_round", how="left")
for col in ["has_wm", "has_sel"]:
    aug[col] = aug[col].fillna(False)
aug["zone"] = aug["round_start_time"].dt.tz_convert(None).dt.date.apply(_assign_zone)

aug["cat_only_wm"]  =  aug["has_wm"] & ~aug["has_sel"]
aug["cat_only_sel"] = ~aug["has_wm"] &  aug["has_sel"]
aug["cat_both"]     =  aug["has_wm"] &  aug["has_sel"]
aug["cat_neither"]  = ~aug["has_wm"] & ~aug["has_sel"]

tb_cats = [
    ("Only WM",  "cat_only_wm"),
    ("Only Sel", "cat_only_sel"),
    ("Both",     "cat_both"),
    ("Neither",  "cat_neither"),
]

zone_order = [label for _, _, label in ZONE_LABELS]
col_w = 24

for winner in ["Wintermute", "Selini", "Kairos"]:
    w_mask = aug["winner_name"] == winner
    print(f"=== {winner} wins ===")
    print(f"{'TB presence':<14}", end="")
    for z in zone_order:
        print(f"  {z.replace(chr(10), ' '):>{col_w}}", end="")
    print()
    print("-" * (14 + (col_w + 2) * len(zone_order)))
    for cat_label, cat_col in tb_cats:
        print(f"{cat_label:<14}", end="")
        for z in zone_order:
            z_mask   = aug["zone"] == z
            n_entity = (w_mask & z_mask).sum()
            n_case   = (w_mask & z_mask & aug[cat_col]).sum()
            pct      = 100 * n_case / n_entity if n_entity > 0 else float("nan")
            print(f"  {f'{pct:.1f}% ({n_case}/{n_entity})':>{col_w}}", end="")
        print()
    print()

## Bid Revenue & Searcher Presence
Daily Arbitrum revenue from Timeboost, bid gap dynamics, and TB presence breakdown by auction winner.

In [ ]:
# Daily total paid bid = Arbitrum revenue from Timeboost
daily_paid = (
    auctions.groupby("date")["paid_bid_usd"]
    .sum()
    .reset_index()
    .rename(columns={"paid_bid_usd": "daily_paid_usd"})
)

df_pv = daily_paid.merge(daily_vol, on="date", how="inner").dropna()



df_pv["zone"] = df_pv["date"].apply(_assign_zone)

fig, ax = plt.subplots(figsize=(8, 5))

print("Arbitrum Revenue (Daily Total Paid Bid) vs ETH Daily Volatility - by Zone")
print(f"{'Zone':<32}  {'r':>7}  {'p':>8}  n")
print("-" * 57)

for (z_start, z_end, z_label), z_color in zip(ZONE_LABELS, ZONE_COLORS):
    sub = df_pv[df_pv["zone"] == z_label]
    if len(sub) < 2:
        continue
    label_flat = z_label.replace("\n", " ")
    ax.scatter(sub["daily_vol"], sub["daily_paid_usd"],
               label=label_flat, color=z_color, edgecolors="0.35",
               linewidths=0.5, s=60, alpha=0.9, zorder=3)
    m, b = np.polyfit(sub["daily_vol"], sub["daily_paid_usd"], 1)
    xs = np.linspace(sub["daily_vol"].min(), sub["daily_vol"].max(), 100)
    ax.plot(xs, m * xs + b, color=z_color, linewidth=1.8)
    r, p = scipy.stats.pearsonr(sub["daily_vol"], sub["daily_paid_usd"])
    print(f"{label_flat:<32}  r={r:+.3f}  p={p:.3g}  n={len(sub)}")

ax.set_xlabel("ETH Daily Volatility")
ax.set_ylabel("Daily Total Paid Bid (USD)")
ax.set_title("Arbitrum Revenue vs ETH Volatility - by Zone")
ax.legend(fontsize=FONT["legend"])
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("figures/paid_bid_vs_vol_by_zone.pdf", bbox_inches="tight")
plt.show()

In [ ]:

# Correlation analysis: Paid Bid, Top Bid, Absolute Bid Gap vs ETH Volatility — by zone
import glob

daily_top = (
    auctions.groupby("date")["top_bid_usd"]
    .sum()
    .reset_index()
    .rename(columns={"top_bid_usd": "daily_top_usd"})
)

# Daily median absolute bid gap (1st - 2nd, in ETH)
_bid_files = sorted(glob.glob("data/case_study/bids/bids_*.csv"))
_bids2 = pd.concat([pd.read_csv(f) for f in _bid_files], ignore_index=True)
_round_times2 = auctions[["auction_round", "round_start_time"]].copy()
_rst2 = pd.to_datetime(_round_times2["round_start_time"])
_round_times2["round_start_time"] = _rst2.dt.tz_convert(None) if _rst2.dt.tz is not None else _rst2

def _fs(amounts):
    s = sorted(amounts, reverse=True)
    if len(s) < 2:
        return pd.Series({"gap_abs": float("nan")})
    return pd.Series({"gap_abs": s[0] - s[1]})   # ETH

_gap2 = _bids2.groupby("Round")["Amount"].apply(_fs).reset_index()
_gap2.columns = ["Round", "metric", "value"]
_gap2 = _gap2.pivot(index="Round", columns="metric", values="value").reset_index()
_gap2 = _gap2.merge(_round_times2, left_on="Round", right_on="auction_round", how="left")
_gap2 = _gap2.dropna(subset=["round_start_time", "gap_abs"])

_rst_g = pd.to_datetime(_gap2["round_start_time"])
_rst_g = _rst_g.dt.tz_convert(None) if _rst_g.dt.tz is not None else _rst_g

daily_gap_med = (
    _gap2.assign(date=_rst_g.dt.date)
    .groupby("date")["gap_abs"].median()
    .reset_index(name="daily_gap_abs_eth")
)

df_all = (
    daily_paid
    .merge(daily_top,     on="date", how="inner")
    .merge(daily_gap_med, on="date", how="inner")
    .merge(daily_vol,     on="date", how="inner")
    .dropna()
)
df_all["zone"] = df_all["date"].apply(_assign_zone)

analyses = [
    ("daily_paid_usd",    "Paid Bid (USD) vs ETH Vol"),
    ("daily_top_usd",     "Top Bid (USD) vs ETH Vol"),
    ("daily_gap_abs_eth", "Absolute Bid Gap (ETH) vs ETH Vol"),
]

for col, title in analyses:
    print(f"\n{'-'*55}")
    print(f"  {title}")
    print(f"  {'Zone':<30}  {'r':>7}  {'p':>9}  n")
    print(f"  {'-'*51}")
    for (_, _, z_label), _ in zip(ZONE_LABELS, ZONE_COLORS):
        sub = df_all[df_all["zone"] == z_label]
        if len(sub) < 2:
            continue
        r, p = scipy.stats.pearsonr(sub["daily_vol"], sub[col])
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"  {z_label.replace(chr(10),' '):<30}  r={r:+.3f}  p={p:.3g}{stars:<4}  n={len(sub)}")


In [ ]:

# Correlation analysis — Paid Bid, Top Bid, Absolute Bid Gap all in ETH vs ETH Vol
import glob

daily_paid_eth = (
    auctions.groupby("date")["paid_bid_eth"]
    .sum().reset_index().rename(columns={"paid_bid_eth": "daily_paid_eth"})
)
daily_top_eth = (
    auctions.groupby("date")["top_bid_eth"]
    .sum().reset_index().rename(columns={"top_bid_eth": "daily_top_eth"})
)

# Absolute bid gap: 1st - 2nd price per round, daily median, in ETH
_bid_files = sorted(glob.glob("data/case_study/bids/bids_*.csv"))
_bids3 = pd.concat([pd.read_csv(f) for f in _bid_files], ignore_index=True)
_rt3 = auctions[["auction_round", "round_start_time"]].copy()
_rst3 = pd.to_datetime(_rt3["round_start_time"])
_rt3["round_start_time"] = _rst3.dt.tz_convert(None) if _rst3.dt.tz is not None else _rst3

def _fs3(amounts):
    s = sorted(amounts, reverse=True)
    if len(s) < 2:
        return pd.Series({"gap_abs": float("nan")})
    return pd.Series({"gap_abs": s[0] - s[1]})

_g3 = _bids3.groupby("Round")["Amount"].apply(_fs3).reset_index()
_g3.columns = ["Round", "metric", "value"]
_g3 = _g3.pivot(index="Round", columns="metric", values="value").reset_index()
_g3 = _g3.merge(_rt3, left_on="Round", right_on="auction_round", how="left").dropna(subset=["round_start_time", "gap_abs"])
_rst_g3 = pd.to_datetime(_g3["round_start_time"])
_rst_g3 = _rst_g3.dt.tz_convert(None) if _rst_g3.dt.tz is not None else _rst_g3

daily_gap_eth = (
    _g3.assign(date=_rst_g3.dt.date)
    .groupby("date")["gap_abs"].median()
    .reset_index(name="daily_gap_eth")
)

df_eth = (
    daily_paid_eth
    .merge(daily_top_eth, on="date", how="inner")
    .merge(daily_gap_eth, on="date", how="inner")
    .merge(daily_vol,     on="date", how="inner")
    .dropna()
)
df_eth["zone"] = df_eth["date"].apply(_assign_zone)

for col, title in [
    ("daily_paid_eth", "Paid Bid (ETH) vs ETH Vol"),
    ("daily_top_eth",  "Top Bid (ETH) vs ETH Vol"),
    ("daily_gap_eth",  "Absolute Bid Gap (ETH) vs ETH Vol"),
]:
    print(f"\n{'-'*55}")
    print(f"  {title}")
    print(f"  {'Zone':<30}  {'r':>7}  {'p':>9}  n")
    print(f"  {'-'*51}")
    for (_, _, z_label), _ in zip(ZONE_LABELS, ZONE_COLORS):
        sub = df_eth[df_eth["zone"] == z_label]
        if len(sub) < 2:
            continue
        r, p = scipy.stats.pearsonr(sub["daily_vol"], sub[col])
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"  {z_label.replace(chr(10),' '):<30}  r={r:+.3f}  p={p:.3g}{stars:<4}  n={len(sub)}")


In [ ]:
import matplotlib.dates as mdates

#  Daily value paid to Kairos (kairos_payments.csv, value / 1e18 = ETH)
kp = pd.read_csv("data/case_study/kairos_payments.csv")
kp["block_time"] = pd.to_datetime(kp["block_time"], utc=True)
kp["date"]       = kp["block_time"].dt.tz_convert(None).dt.normalize()
kp["value_eth"]  = kp["value"] / 1e18

print(f"Total value received by Kairos: {kp['value_eth'].sum():,.2f} ETH")
print(f"Total payment count: {len(kp)}")

kp_daily = kp.groupby("date")["value_eth"].sum().reset_index(name="daily_value_eth")

eth_price_daily = (
    binance_ethusd[["timestamp", "ETH_mid"]]
    .assign(date=lambda d: d["timestamp"].dt.tz_convert(None).dt.normalize())
    .groupby("date")["ETH_mid"].mean()
    .reset_index()
)
kp_daily = kp_daily.merge(eth_price_daily, on="date", how="left")
kp_daily["daily_value_usd"] = kp_daily["daily_value_eth"] * kp_daily["ETH_mid"]

# Bid paid by Kairos from auctions
kairos_bids = (
    auctions[auctions["winner_name"] == "Kairos"]
    .assign(date=lambda d: pd.to_datetime(d["round_start_time"]).dt.tz_convert(None).dt.normalize())
    .groupby("date")["paid_bid_usd"].sum()
    .reset_index(name="daily_bid_usd")
)

plot_df = kp_daily.merge(kairos_bids, on="date", how="outer").sort_values("date").fillna(0)

date_min = plot_df["date"].min()
date_max = plot_df["date"].max()

fig, ax = plt.subplots(figsize=FIG["wide_short"])

# Zone shading (before data)
for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_datetime64()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
    ax.axvspan(x0, x1, alpha=0.12, color=ZONE_COLORS[i], zorder=0)
_add_event_lines(ax)

_BAR_W      = pd.Timedelta(hours=9.6)
_BAR_OFFSET = pd.Timedelta(hours=4.8)
ax.bar(plot_df["date"] - _BAR_OFFSET, plot_df["daily_value_usd"],
       width=_BAR_W, color="#2ca02c", alpha=0.85, label="Payments received", zorder=3)
ax.bar(plot_df["date"] + _BAR_OFFSET, plot_df["daily_bid_usd"],
       width=_BAR_W, color="#d62728", alpha=0.85, label="Bids paid", zorder=3)

ax.set_ylabel("USD", fontsize=FONT["label"])
ax.set_axisbelow(True)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(fontsize=FONT["legend"])
ax.tick_params(axis="both", labelsize=FONT["tick"])

ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

# Zone text labels above panel
for z_start, z_end, z_label in ZONE_LABELS:
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax.text(mid, 1.01, z_label, ha="center", va="bottom",
            fontsize=FONT["annot"], color="0.25", style="italic",
            zorder=6, transform=ax.get_xaxis_transform(), clip_on=False)

ax.set_xlim(date_min, date_max)
fig.suptitle("Daily Value Received vs Bid Paid by Kairos", fontsize=FONT["title"])
fig.tight_layout()
plt.savefig("figures/kairos_daily_value.pdf", bbox_inches="tight", dpi=300)
plt.show()

print(f"Payments received : ${plot_df['daily_value_usd'].sum():,.0f} USD")
print(f"Bids paid         : ${plot_df['daily_bid_usd'].sum():,.0f} USD")
print(f"Net               : ${(plot_df['daily_value_usd'] - plot_df['daily_bid_usd']).sum():,.0f} USD")


In [ ]:
# Searcher PnL in rounds where Kairos was the auction winner
h = 5
kairos_rounds = set(auctions[auctions["winner_name"] == "Kairos"]["auction_round"].dropna().astype(int))

pnl_in_kairos = txs[txs["auction_round"].isin(kairos_rounds)].copy()

print(f"Kairos-won rounds : {len(kairos_rounds):,}")
print()
print(f"{'Searcher':<14} {'Pos-PnL TXs':>12} {'Total PnL (pos)':>16}")
print("-" * 45)
total_txs, total_pnl = 0, 0
for name, addrs in SEARCHERS.items():
    sub = pnl_in_kairos[
        pnl_in_kairos["tx_to_address"].isin(addrs) &
        (pnl_in_kairos[f"pnl_t{h}"] > 0) &
        (pnl_in_kairos["timeboosted"] == True)
    ]
    total_txs += len(sub)
    total_pnl += sub[f"pnl_t{h}"].sum()
    print(f"{name:<14} {len(sub):>12,} ${sub[f'pnl_t{h}'].sum():>15,.0f}")
print("-" * 45)
print(f"{'Total':<14} {total_txs:>12,} ${total_pnl:>15,.0f}")


In [ ]:
all_searcher_addrs = {a.lower(): name for name, addrs in SEARCHERS.items() for a in addrs}

kp_check = kp.copy()
kp_check["tx_to_lower"] = kp_check["tx_to"].str.lower()
kp_check["searcher"]    = kp_check["tx_to_lower"].map(all_searcher_addrs)

print("tx_to address breakdown:")
print(kp_check["tx_to"].value_counts().to_string())
print()
print("Matches to known searchers:")
print(kp_check["searcher"].value_counts(dropna=False).to_string())
print()
matched = kp_check["searcher"].notna().sum()
print(f"{matched} / {len(kp_check)} rows ({100*matched/len(kp_check):.1f}%) have tx_to = known searcher")

print(f"Total payment txounts: {len(kp_check)}")

In [ ]:
#  Number of bidders per auction round, over time 
import glob

bid_files = sorted(glob.glob("data/case_study/bids/bids_*.csv"))
bids_all  = pd.concat([pd.read_csv(f) for f in bid_files], ignore_index=True)

print(f"Total bids loaded: {len(bids_all)}")
print(f"Average bids per round: {bids_all.groupby('Round').size().mean():.2f}")
print(f"Total unique bidders: {bids_all['Bidder'].nunique()}")
print(f"Number of unique rounds: {bids_all['Round'].nunique()}")

# Bid share by named bidder (bidder addresses, not searcher/tx addresses)
_bid_addrs = {
    "Wintermute": ["0x8C6FFc5501ACA218f5F60E140d2c2461092Ec3E2"],
    "Selini":     ["0x95c0d21482fd6bc204E588C06632fDb1CF51b018"],
    "Kairos":     ["0x2b38a73dd32a2eafe849825a4b515ae5187eda42"],
}
total_bids = len(bids_all)
print("\nBid share by bidder:")
for name, addrs in _bid_addrs.items():
    n = bids_all["Bidder"].str.lower().isin([a.lower() for a in addrs]).sum()
    print(f"  {name:<12} {n:>6,}  ({n/total_bids:.1%})")
n_other = total_bids - sum(
    bids_all["Bidder"].str.lower().isin([a.lower() for a in addrs]).sum()
    for addrs in _bid_addrs.values()
)
print(f"  {'Other':<12} {n_other:>6,}  ({n_other/total_bids:.1%})")

bidders_per_round = (
    bids_all.groupby("Round")["Bidder"]
    .nunique()
    .reset_index(name="n_bidders")
)

round_times = auctions[["auction_round", "round_start_time"]].copy()
_rst = pd.to_datetime(round_times["round_start_time"])
round_times["round_start_time"] = (
    _rst.dt.tz_convert(None) if _rst.dt.tz is not None else _rst
)
bidders_per_round = bidders_per_round.merge(
    round_times, left_on="Round", right_on="auction_round", how="left"
).dropna(subset=["round_start_time"]).sort_values("round_start_time")

date_min = bidders_per_round["round_start_time"].min()
date_max = bidders_per_round["round_start_time"].max()

fig, ax = plt.subplots(figsize=(14, 4))

for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_datetime64()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
    ax.axvspan(x0, x1, alpha=0.12, color=ZONE_COLORS[i], zorder=0)

# Event lines  14 each with a distinct color for the legend
event_handles = []
for ts, label, color in EVENT_LINES:
    line = ax.axvline(ts.tz_convert(None).to_pydatetime(), color=color,
                      linewidth=1.2, linestyle="--", zorder=6, label=label)
    event_handles.append(line)


ax.scatter(bidders_per_round["round_start_time"], bidders_per_round["n_bidders"],
           s=2, alpha=0.4, color="#1f77b4", linewidths=0)

for z_start, z_end, z_label in ZONE_LABELS:
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax.text(mid, 1.01, z_label, ha="center", va="bottom",
            fontsize=FONT["annot"], color="0.25", style="italic", zorder=6, transform=ax.get_xaxis_transform(), clip_on=False)

ax.set_ylabel("Bidders per Round")
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.set_xlim(date_min, date_max)
ax.set_ylim(bottom=0)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
# ax.legend(handles=event_handles, title="Events", fontsize=FONT["legend"],
#           title_fontsize=FONT["legend"]+1, loc="upper right", framealpha=0.85)
fig.suptitle("Number of Bidders per Auction Round", fontsize=FONT["title"])
fig.tight_layout()
plt.savefig("figures/bidders_per_round.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
ADDR_WM  = "0x8C6FFc5501ACA218f5F60E140d2c2461092Ec3E2".lower()   # Wintermute
ADDR_SEL = "0x95c0d21482fd6bc204E588C06632fDb1CF51b018".lower()   # Selini
ADDR_KAI = "0x2b38a73dd32a2eafe849825a4b515ae5187eda42".lower()   # Kairos

b = bids_all.copy()
b["Bidder_lower"] = b["Bidder"].str.lower()

presence = b.groupby("Round")["Bidder_lower"].apply(set).reset_index(name="bidders")
presence["wm"]  = presence["bidders"].apply(lambda s: ADDR_WM  in s)
presence["sel"] = presence["bidders"].apply(lambda s: ADDR_SEL in s)
presence["kai"] = presence["bidders"].apply(lambda s: ADDR_KAI in s)

presence = presence.merge(round_times, left_on="Round", right_on="auction_round", how="left")
presence["zone"] = presence["round_start_time"].apply(
    lambda t: _assign_zone(t.date()) if pd.notna(t) else None
)

combos = [
    ("WM + Sel + Kai", lambda d: d["wm"]  &  d["sel"] &  d["kai"]),
    ("WM + Sel",       lambda d: d["wm"]  &  d["sel"] & ~d["kai"]),
    ("WM + Kai",       lambda d: d["wm"]  & ~d["sel"] &  d["kai"]),
    ("Sel + Kai",      lambda d: ~d["wm"] &  d["sel"] &  d["kai"]),
    ("Only WM",        lambda d: d["wm"]  & ~d["sel"] & ~d["kai"]),
    ("Only Sel",       lambda d: ~d["wm"] &  d["sel"] & ~d["kai"]),
    ("Only Kai",       lambda d: ~d["wm"] & ~d["sel"] &  d["kai"]),
    ("None",           lambda d: ~d["wm"] & ~d["sel"] & ~d["kai"]),
]

print("Bidder presence per auction round, by zone")
print(f"  WM  = {ADDR_WM}  (Wintermute)")
print(f"  Sel = {ADDR_SEL}  (Selini)")
print(f"  Kai = {ADDR_KAI}  (Kairos)")
print()

zones      = [lbl.replace("\n", " ") for _, _, lbl in ZONE_LABELS]
zones_raw  = [lbl for _, _, lbl in ZONE_LABELS]
combo_w    = 16
# col wide enough for both header zone name and content like "100.0% (9999/9999)"
col_w      = max(max(len(z) for z in zones), len("100.0% (9999/9999)")) + 2

sep = "-" * (combo_w + (col_w + 2) * (len(zones) + 1))
header = f"{'Combination':<{combo_w}}" + "".join(f"  {z:^{col_w}}" for z in zones) + f"  {'Overall':^{col_w}}"
print(header)
print(sep)

for label, mask_fn in combos:
    row = f"{label:<{combo_w}}"
    total_n = total_match = 0
    for zone_raw in zones_raw:
        sub = presence[presence["zone"] == zone_raw]
        n   = len(sub)
        m   = mask_fn(sub).sum()
        pct = 100 * m / n if n else float("nan")
        cell = f"{pct:.1f}% ({m}/{n})"
        row += f"  {cell:^{col_w}}"
        total_n += n; total_match += m
    pct_all = 100 * total_match / total_n if total_n else float("nan")
    cell = f"{pct_all:.1f}% ({total_match}/{total_n})"
    row += f"  {cell:^{col_w}}"
    print(row)

print(sep)

In [ ]:
orphan_rounds = set(presence[presence["round_start_time"].isna()]["Round"])
n_orphan_bids = bids_all[bids_all["Round"].isin(orphan_rounds)].shape[0]
print(f"Total bids:              {len(bids_all):,}")
print(f"  in zoned rounds:       {len(bids_all[~bids_all['Round'].isin(orphan_rounds)]):,}")
print(f"  in orphan rounds:      {n_orphan_bids:,}")
effective_rounds = presence[presence["round_start_time"].notna()]["Round"].nunique()
print(f"Effective rounds: {effective_rounds:,}")


In [ ]:
#  First-price / second-price gap per round, normalized
# For each round: sort bids desc, gap = (1st - 2nd) / 1st
def first_second(amounts):
    s = sorted(amounts, reverse=True)
    if len(s) < 2:
        return pd.Series({"first": s[0], "second": float("nan"), "gap_norm": float("nan")})
    return pd.Series({"first": s[0], "second": s[1],
                      "gap_norm": (s[0] - s[1]) / s[0] if s[0] > 0 else float("nan")})

gap_df = (
    bids_all.groupby("Round")["Amount"]
    .apply(first_second)
    .reset_index()
)
gap_df.columns = ["Round", "metric", "value"]
gap_df = gap_df.pivot(index="Round", columns="metric", values="value").reset_index()

gap_df = gap_df.merge(round_times, left_on="Round", right_on="auction_round", how="left")
gap_df = gap_df.dropna(subset=["round_start_time", "gap_norm"]).sort_values("round_start_time")

# Daily median for a summary line
daily_gap = (
    gap_df.assign(date=lambda d: d["round_start_time"].dt.normalize())
    .groupby("date")["gap_norm"]
    .median()
    .reset_index(name="median_gap_norm")
)

date_min = gap_df["round_start_time"].min()
date_max = gap_df["round_start_time"].max()

fig, ax = plt.subplots(figsize=(14, 4))

for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_datetime64()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
    ax.axvspan(x0, x1, alpha=0.12, color=ZONE_COLORS[i], zorder=0)
_add_event_lines(ax)

ax.scatter(gap_df["round_start_time"], gap_df["gap_norm"],
           s=1.5, alpha=0.25, color="#1f77b4", linewidths=0, label="Per-round")
ax.plot(daily_gap["date"], daily_gap["median_gap_norm"],
        color="black", linewidth=1.8, label="Daily median", zorder=5)

ax.set_ylabel("(1st - 2nd) / 1st")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_ylim(0, 1)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(fontsize=FONT["legend"])

for z_start, z_end, z_label in ZONE_LABELS:
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax.text(mid, 1.01, z_label, ha="center", va="bottom",
            fontsize=7.5, color="0.25", style="italic", zorder=6, transform=ax.get_xaxis_transform(), clip_on=False)

plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.suptitle("Normalized Bid Gap per Auction Round", fontsize=FONT["title"])
fig.tight_layout()
plt.savefig("figures/bid_gap_normalized.pdf", bbox_inches="tight", dpi=300)
plt.show()

print("Rounds with only 1 bidder (no 2nd price):",
      gap_df["second"].isna().sum(), "/", len(gap_df))

In [ ]:
auctions["date"] = auctions["round_start_time"].dt.date
auctions["bid_gap"] = auctions["top_bid_usd"] - auctions["paid_bid_usd"]
daily_bid_gap = auctions.groupby("date")["bid_gap"].mean().reset_index()
daily_gap_vol = daily_bid_gap.merge(daily_vol, on="date", how="inner")
df = daily_gap_vol.dropna(subset=["bid_gap", "daily_vol"])

pearson_r, pearson_p = scipy.stats.pearsonr(df["bid_gap"], df["daily_vol"])
spearman_r, spearman_p = scipy.stats.spearmanr(df["bid_gap"], df["daily_vol"])

print("Correlation between Daily Bid Gap and Daily Volatility:")
print(f"Pearson:  r={pearson_r:.4f}, p={pearson_p:.4g}")
print(f"Spearman: r={spearman_r:.4f}, p={spearman_p:.4g}")

In [ ]:
# Daily mean bid gap vs ETH volatility, by zone
df_bg = daily_bid_gap.merge(daily_vol, on="date", how="inner").dropna()
df_bg["zone"] = df_bg["date"].apply(_assign_zone)

fig, ax = plt.subplots(figsize=(8, 5))

print("Daily Mean Bid Gap vs ETH Daily Volatility by Zone")
print(f"{'Zone':<32}  {'r':>7}  {'p':>8}  n")
print("-" * 57)

for (z_start, z_end, z_label), z_color in zip(ZONE_LABELS, ZONE_COLORS):
    sub = df_bg[df_bg["zone"] == z_label]
    if len(sub) < 2:
        continue
    label_flat = z_label.replace("\n", " ")
    ax.scatter(sub["daily_vol"], sub["bid_gap"],
               label=label_flat, color=z_color, edgecolors="0.35",
               linewidths=0.5, s=60, alpha=0.9, zorder=3)
    m, b = np.polyfit(sub["daily_vol"], sub["bid_gap"], 1)
    xs = np.linspace(sub["daily_vol"].min(), sub["daily_vol"].max(), 100)
    ax.plot(xs, m * xs + b, color=z_color, linewidth=1.8)
    r, p = scipy.stats.pearsonr(sub["daily_vol"], sub["bid_gap"])
    print(f"{label_flat:<32}  r={r:+.3f}  p={p:.3g}  n={len(sub)}")

ax.set_xlabel("ETH Daily Volatility")
ax.set_ylabel("Daily Mean Bid Gap (USD)")
ax.set_title("Bid Gap vs ETH Volatility by Zone")
ax.legend(fontsize=FONT["legend"])
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("figures/bid_gap_vs_vol_by_zone.pdf", bbox_inches="tight")
plt.show()

In [ ]:
def compute_daily_pnl_gaps(txs, daily_vol):
    """
    txs: filtered tx dataset for a single actor (wm_txs, sel_txs, ...)
    daily_vol: dataframe with "date" and "daily_vol"
    
    Returns: dict of {horizon -> merged df with tb, ntb, gap, vol}
    """
    if "date" not in txs:
        txs = txs.copy()
        txs["date"] = txs["block_time"].dt.date

    daily_summary = {}

    for h in HORIZONS:
        col = f"pnl_t{h}"
        df_pos = txs[txs[col] > 0]

        df = (
            df_pos.groupby(["date", "timeboosted"])[col]
            .sum()
            .reset_index()
            .rename(columns={col: f"pnl_t{h}"})
        )

        pivot = (
            df.pivot(index="date", columns="timeboosted", values=f"pnl_t{h}")
              .rename(columns={True: f"tb_pnl_t{h}", False: f"ntb_pnl_t{h}"})
              .reset_index()
        )

        merged = pivot.merge(daily_vol, on="date", how="left")
        merged[f"gap_t{h}"] = merged[f"tb_pnl_t{h}"] - merged[f"ntb_pnl_t{h}"]

        daily_summary[h] = merged

    return daily_summary

In [ ]:
wm_summary = compute_daily_pnl_gaps(wm_txs, daily_vol)
sel_summary = compute_daily_pnl_gaps(sel_txs, daily_vol)

## Daily PnL Over Time (t=5s Markout)
Timeboosted vs Non-Timeboosted PnL per searcher, using 5-second markout horizon.

In [ ]:
def _add_vol_axis(ax, df, date_col="date"):
    """Overlay daily_vol on a twin y-axis."""
    ax2 = ax.twinx()
    ax2.plot(df[date_col], df["daily_vol"],
             color="gray", linewidth=2.2, linestyle=":", label="Volatility")
    ax2.set_ylabel("ETH Daily Volatility", color="gray", fontsize=FONT["label"])
    ax2.tick_params(axis="y", labelcolor="gray")
    ax2.yaxis.set_label_position("right")
    return ax2


def _shade_zones(ax, date_min, date_max):
    for i, (z_start, z_end, z_label) in enumerate(ZONE_LABELS):
        x0 = date_min if z_start is None else z_start.tz_convert(None).to_datetime64()
        x1 = date_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
        ax.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)
        mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
        ax.text(mid, 1.01, z_label, ha="center", va="bottom",
                fontsize=7.5, color="0.3", style="italic", transform=ax.get_xaxis_transform(), clip_on=False)
    _add_event_lines(ax)

h = 5

# -- Build per-searcher daily PnL ---------------------------------------------
_searcher_data = []
for name, summary, color in [
    ("Wintermute", wm_summary,  "#1f77b4"),
    ("Selini",     sel_summary, "#ff7f0e"),
]:
    df = summary[h].copy()
    df["date"] = pd.to_datetime(df["date"])
    full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    df = df.set_index("date").reindex(full_range).reset_index().rename(columns={"index": "date"})
    _searcher_data.append((name, df, color))

t_min = min(d["date"].min() for _, d, _ in _searcher_data)
t_max = max(d["date"].max() for _, d, _ in _searcher_data)

fig, ax = plt.subplots(figsize=FIG["wide"])

# Zone shading (before data)
for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_datetime64()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
    ax.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)

for name, df, color in _searcher_data:
    ax.plot(df["date"], df[f"tb_pnl_t{h}"],  color=color, linewidth=1.8,
            marker="o", markersize=3, label=f"{name} TB")
    ax.plot(df["date"], df[f"ntb_pnl_t{h}"], color=color, linewidth=1.8,
            marker="s", markersize=3, linestyle="--", alpha=0.6, label=f"{name} Non-TB")

ax.axhline(0, color="black", linewidth=0.6, linestyle=":")
_add_event_lines(ax)


# Zone text labels (after data â†’ correct ylim)
for z_start, z_end, z_label in ZONE_LABELS:
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax.text(mid, 1.01, z_label, ha="center", va="bottom",
            fontsize=FONT["annot"], color="0.25", style="italic", zorder=6, transform=ax.get_xaxis_transform(), clip_on=False)

ax.legend(fontsize=FONT["legend"])

ax.set_ylabel("Daily PnL (USD, positive trades only)")
ax.set_xlim(t_min, t_max)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

fig.suptitle(f"Daily Total PnL — {h}s Markout (TB vs Non-TB)", fontsize=FONT["title"])
fig.tight_layout()
plt.savefig("figures/daily_pnl_over_time_t5.pdf", bbox_inches="tight", dpi=300)
plt.show()


## Average PnL Per Trade Per Day (t=5s Markout)
Mean per-trade PnL (positive trades only) by day, split by TB vs Non-TB and searcher.

In [ ]:
h = 5
pnl_col = f"pnl_t{h}"

# -- Build per-searcher daily avg PnL -----------------------------------------
_avg_data = []
for name, addrs, color in [
    ("Wintermute", SEARCHERS["Wintermute"], "#1f77b4"),
    ("Selini",     SEARCHERS["Selini"],     "#ff7f0e"),
]:
    df = txs[txs["tx_to_address"].isin(addrs)].copy()
    df["date"] = pd.to_datetime(df["block_time"]).dt.date

    avg = (
        df[df[pnl_col] > 0]
        .groupby(["date", "timeboosted"])[pnl_col]
        .mean()
        .unstack(fill_value=np.nan)
        .rename(columns={True: "tb_avg", False: "ntb_avg"})
        .reset_index()
    )
    avg["date"] = pd.to_datetime(avg["date"])
    full_range = pd.date_range(avg["date"].min(), avg["date"].max(), freq="D")
    avg = avg.set_index("date").reindex(full_range).reset_index().rename(columns={"index": "date"})
    _avg_data.append((name, avg, color))

t_min = min(d["date"].min() for _, d, _ in _avg_data)
t_max = max(d["date"].max() for _, d, _ in _avg_data)

fig, ax = plt.subplots(figsize=FIG["wide"])

# Zone shading (before data)
for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_datetime64()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
    ax.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)

for name, avg, color in _avg_data:
    ax.plot(avg["date"], avg["tb_avg"],  color=color, linewidth=1.8,
            marker="o", markersize=3, label=f"{name} TB")
    ax.plot(avg["date"], avg["ntb_avg"], color=color, linewidth=1.8,
            marker="s", markersize=3, linestyle="--", alpha=0.6, label=f"{name} Non-TB")

ax.axhline(0, color="black", linewidth=0.6, linestyle=":")
_add_event_lines(ax)


# Zone text labels (after data â†’ correct ylim)
for z_start, z_end, z_label in ZONE_LABELS:
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax.text(mid, 1.01, z_label, ha="center", va="bottom",
            fontsize=FONT["annot"], color="0.25", style="italic", zorder=6, transform=ax.get_xaxis_transform(), clip_on=False)

ax.legend(fontsize=FONT["legend"])

ax.set_ylabel("Avg PnL per trade (USD, positive trades only)")
ax.set_xlim(t_min, t_max)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

fig.suptitle(f"Daily Avg PnL per Trade — {h}s Markout (TB vs Non-TB)", fontsize=FONT["title"])
fig.tight_layout()
plt.savefig("figures/daily_avg_pnl_per_trade_t5.pdf", bbox_inches="tight", dpi=300)
plt.show()


## Total Surplus Over Time


In [ ]:
h = 5
only_pos = True

#  1. Daily searcher PnL
if only_pos:
    searcher_pnl = txs[txs[f"pnl_t{h}"] > 0].copy()
else:
    searcher_pnl = txs.copy()
searcher_pnl["date"] = pd.to_datetime(searcher_pnl["block_time"]).dt.date
daily_pnl = searcher_pnl.groupby("date")[f"pnl_t{h}"].sum().rename("total_pnl")

#  2. Daily tx fees
if only_pos:
    fees = txs[txs[f"pnl_t{h}"] > 0].copy()
else:
    fees = txs.copy()
fees["date"] = pd.to_datetime(fees["block_time"]).dt.date
daily_fees = fees.groupby("date")["tx_fee_usd"].sum().rename("total_fees")

#  3. Daily bid paid (second-price, actual) and first-price hypothetical
auctions_s = auctions.copy()
auctions_s["date"] = auctions_s["round_start_time"].dt.tz_convert(None).dt.date
daily_bids    = auctions_s.groupby("date")["paid_bid_usd"].sum().rename("total_bids_paid")
daily_bids_fp = auctions_s.groupby("date")["top_bid_usd"].sum().rename("total_bids_fp")

#  4. Merge and compute surplus
surplus = (
    pd.DataFrame({"date": pd.date_range(
        start=min(daily_pnl.index.min(), daily_fees.index.min(), daily_bids.index.min()),
        end=max(daily_pnl.index.max(), daily_fees.index.max(), daily_bids.index.max()),
        freq="D",
    ).date})
    .set_index("date")
    .join(daily_pnl, how="left")
    .join(daily_fees, how="left")
    .join(daily_bids, how="left")
    .join(daily_bids_fp, how="left")
    .fillna(0)
    .reset_index()
)
surplus["date"] = pd.to_datetime(surplus["date"])
surplus["surplus"] = surplus["total_pnl"] - surplus["total_fees"] - surplus["total_bids_paid"]

total = surplus["total_pnl"].replace(0, float("nan"))
surplus["pct_surplus"] = surplus["surplus"]         / total * 100
surplus["pct_bids"]    = surplus["total_bids_paid"] / total * 100
surplus["pct_fees"]    = surplus["total_fees"]      / total * 100

# First-price hypothetical
surplus["pct_bids_fp"]    = surplus["total_bids_fp"] / total * 100
surplus["pct_surplus_fp"] = 100 - surplus["pct_fees"] - surplus["pct_bids_fp"]

_vol_idx = pd.Series(
    daily_vol["daily_vol"].values,
    index=pd.to_datetime(daily_vol["timestamp"])
)
surplus["daily_vol"] = surplus["date"].map(_vol_idx)

#  5. Plot
date_min = surplus["date"].min()
date_max = surplus["date"].max()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=FIG["wide_tall"], sharex=True)

# Zone shading (before data)
for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_datetime64()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
    ax1.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)
    ax2.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)

ax1.bar(surplus["date"], surplus["total_pnl"],        label="Searcher PnL", color="#1f77b4", alpha=0.7, width=0.8)
ax1.bar(surplus["date"], -surplus["total_fees"],      label="-Tx Fees",     color="#ff7f0e", alpha=0.7, width=0.8)
ax1.bar(surplus["date"], -surplus["total_bids_paid"], label="-Bids Paid",   color="#d62728", alpha=0.7, width=0.8,
        bottom=-surplus["total_fees"])
ax1.plot(surplus["date"], surplus["surplus"], color="black", linewidth=2.2,
         marker="D", markersize=4, label="Net Surplus", zorder=5)
ax1.axhline(0, color="black", linewidth=0.6, linestyle="-.")
ax1.set_ylabel("USD")
ax1.set_axisbelow(True)
ax1.grid(axis="y", linestyle="--", alpha=0.4)
_add_event_lines(ax1)

ax1v = ax1.twinx()
ax1v.plot(surplus["date"], surplus["daily_vol"],
          color="grey", linewidth=1.8, linestyle=":", label="Volatility", zorder=6)
ax1v.set_ylabel("ETH Daily Volatility", color="grey", fontsize=FONT["label"])
ax1v.tick_params(axis="y", labelcolor="grey")

lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax1v.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, fontsize=FONT["legend"], loc="upper right")

# Zone text labels for ax1 (after data â†’ correct ylim)
for z_start, z_end, z_label in ZONE_LABELS:
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax1.text(mid, 1.01, z_label, ha="center", va="bottom",
             fontsize=FONT["annot"], color="0.25", style="italic", zorder=6, transform=ax1.get_xaxis_transform(), clip_on=False)

# Bottom: 100% stacked area — soft muted palette
dates = surplus["date"]
y1 = surplus["pct_surplus"]
y2 = surplus["pct_surplus"] + surplus["pct_bids"]
y3 = surplus["pct_surplus"] + surplus["pct_bids"] + surplus["pct_fees"]

ax2.set_facecolor("#f4f4f4")  # light gray panel background

# Very light fills to indicate areas
ax2.fill_between(dates, 0,  y1, color="#aec7e8", alpha=0.45, zorder=1, label="Net Surplus (2nd price)")
ax2.fill_between(dates, y1, y3, color="#f4a582", alpha=0.45, zorder=1, label="Bids Paid + Tx Fees")

# Boundary lines to delineate areas clearly
ax2.plot(dates, y1, color="#2c7bb6", linewidth=1.6, zorder=4)

# Hypothetical first-price: hatch the gap between 2nd and 1st price surplus
ax2.fill_between(dates, surplus["pct_surplus_fp"], y1,
                 color="#d7191c", alpha=0.12, hatch="///", linewidth=0,
                 zorder=3, label="Capturable gap (1st−2nd price)")
ax2.plot(dates, surplus["pct_surplus_fp"], color="#2c7bb6", linewidth=1.8,
         linestyle="--", zorder=6, label="Net Surplus (1st price)")

ax2.set_ylim(0, 100)
ax2.set_ylabel("% of PnL")
ax2.set_axisbelow(True)
ax2.grid(axis="y", linestyle="--", alpha=0.35, color="white", zorder=2)
ax2.legend(fontsize=FONT["legend"]-1, loc="lower left")
_add_event_lines(ax2)

# Zone boundary lines only for ax2 (no text labels — already shown in top panel)
for z_start, z_end, z_label in ZONE_LABELS:
    if z_start is not None:
        x0 = z_start.tz_convert(None).to_pydatetime()
        ax2.axvline(x0, color="0.4", linewidth=1.5, linestyle="--", zorder=5)

ax2.set_xlim(date_min, date_max)
ax2.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.setp(ax2.get_xticklabels(), rotation=30, ha="right")

title_suffix = "Positive PnL Trades Only" if only_pos else "All Trades"
fig.suptitle(f"Total Daily Surplus at {h}s Markout ({title_suffix})", fontsize=FONT["title"])
fig.tight_layout()
plt.savefig(f"figures/total_surplus_over_time_{h}s.pdf", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
# Bids paid as % of PnL by zone (uses surplus df from cell above)
_s = surplus.copy()
_s["zone"] = _s["date"].apply(lambda d: _assign_zone(d.date()))

print(f"{'Zone':<30} {'Mean 2nd':>9} {'Med 2nd':>8} {'Mean 1st':>9} {'Med 1st':>8} {'n':>5}")
print("-" * 78)
for _, _, z_label in ZONE_LABELS:
    sub = _s[_s["zone"] == z_label]
    sp = sub["pct_bids"].replace([float("inf"), float("-inf")], float("nan")).dropna()
    fp = sub["pct_bids_fp"].replace([float("inf"), float("-inf")], float("nan")).dropna()
    n = len(sp)
    print(f"{z_label.replace(chr(10), ' '):<30} {sp.mean():>8.1f}% {sp.median():>7.1f}% {fp.mean():>8.1f}% {fp.median():>7.1f}% {n:>5}")


In [ ]:
from scipy import stats

df_corr = surplus.dropna(subset=["daily_vol", "surplus", "total_pnl"]).copy()
df_corr["zone"] = df_corr["date"].apply(_assign_zone)

def _corr_row(x, y, label_x, label_y, subset="All"):
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    r, p_r = stats.pearsonr(x, y)
    rho, p_rho = stats.spearmanr(x, y)
    return dict(subset=subset, x=label_x, y=label_y,
                n=len(x), pearson_r=r, p_pearson=p_r,
                spearman_rho=rho, p_spearman=p_rho)

rows = []
for y_col, y_label in [("surplus", "Net Surplus"), ("total_pnl", "Searcher PnL")]:
    rows.append(_corr_row(df_corr["daily_vol"], df_corr[y_col], "ETH Daily Vol", y_label))
    for zone, grp in df_corr.groupby("zone"):
        rows.append(_corr_row(grp["daily_vol"], grp[y_col], "ETH Daily Vol", y_label, subset=zone))

df_res = (pd.DataFrame(rows)[["y", "subset", "n", "pearson_r", "p_pearson", "spearman_rho", "p_spearman"]]
          .round({"pearson_r": 6, "p_pearson": 6, "spearman_rho": 3, "p_spearman": 3}))
print("Correlation: ETH Daily Volatility vs Surplus / PnL")
print(df_res.to_string(index=False))

#  Scatter plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (y_col, y_label) in zip(axes, [("surplus", "Net Surplus (USD)"), ("total_pnl", "Searcher PnL (USD)")]):
    for (z_start, z_end, z_label), z_color in zip(ZONE_LABELS, ZONE_COLORS):
        sub = df_corr[df_corr["zone"] == z_label]
        ax.scatter(sub["daily_vol"], sub[y_col], label=z_label, color=z_color,
                   edgecolors="0.35", linewidths=0.5, alpha=0.8, s=40)
    x_all = df_corr["daily_vol"].values
    y_all = df_corr[y_col].values
    m, b, r, p, _ = stats.linregress(x_all, y_all)
    x_line = np.linspace(x_all.min(), x_all.max(), 200)
    ax.plot(x_line, m * x_line + b, color="black", linewidth=1.5, linestyle="--",
            label=f"OLS (r={r:.2f}, p={p:.6f})")
    ax.axhline(0, color="0.5", linewidth=0.6, linestyle=":")
    ax.set_xlabel("ETH Daily Volatility")
    ax.set_ylabel(y_label)
    ax.set_title(y_label)
    ax.legend(fontsize=FONT["legend"])
    ax.grid(linestyle="--", alpha=0.4)

fig.suptitle(f"ETH Volatility vs Surplus / Searcher PnL (t={h}s markout)", fontsize=FONT["title"])
fig.tight_layout()
plt.savefig(f"figures/surplus_vol_correlation_{h}s_{title_suffix.strip(' ()')}.pdf", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
_BAR_W      = pd.Timedelta(hours=9.6)
_BAR_OFFSET = pd.Timedelta(hours=4.8)

fig, axes = plt.subplots(1, 2, figsize=FIG["double"], sharey=False)

for ax, (name, addrs, base_color) in zip(axes, [
    ("Wintermute", SEARCHERS["Wintermute"], "#1f77b4"),
    ("Selini",     SEARCHERS["Selini"],     "#ff7f0e"),
]):
    df = txs[txs["tx_to_address"].isin(addrs)].copy()
    df["date"] = pd.to_datetime(df["block_time"].dt.date)

    daily = (
        df.groupby(["date", "timeboosted"]).size()
        .unstack(fill_value=0)
        .rename(columns={True: "TB", False: "Non-TB"})
        .reset_index()
    )
    daily["date"] = pd.to_datetime(daily["date"])
    t_min, t_max = daily["date"].min(), daily["date"].max()

    # Zone shading (before data)
    for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
        x0 = t_min if z_start is None else z_start.tz_convert(None).to_datetime64()
        x1 = t_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
        ax.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)

    ax.bar(daily["date"] - _BAR_OFFSET, daily["TB"],     width=_BAR_W,
           color=base_color, alpha=0.85, label="Timeboosted", zorder=3)
    ax.bar(daily["date"] + _BAR_OFFSET, daily["Non-TB"], width=_BAR_W,
           color=base_color, alpha=0.35, label="Regular", zorder=3)

    _add_event_lines(ax)

    # Zone text labels (after data â†’ correct ylim)
    for z_start, z_end, z_label in ZONE_LABELS:
        x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
        x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
        mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
        ax.text(mid, 1.01, z_label, ha="center", va="bottom",
        fontsize=10, color="0.25", style="italic", zorder=6, transform=ax.get_xaxis_transform(), clip_on=False)

    ax.set_title(name, fontsize=FONT["title"])
    ax.set_ylabel("Transaction count")
    ax.set_xlim(t_min - _BAR_W, t_max + _BAR_W)
    ax.set_axisbelow(True)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.legend(fontsize=FONT["legend"])

fig.suptitle("Daily Timeboosted and Regular CEX-DEX Transaction Count per Searcher", fontsize=FONT["title"])
fig.tight_layout()
plt.savefig("figures/daily_tx_count_tb_vs_ntb.pdf", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
_BAR_W      = pd.Timedelta(hours=9.6)
_BAR_OFFSET = pd.Timedelta(hours=4.8)
h = 5
pnl_col = f"pnl_t{h}"

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
(ax_tl, ax_tr), (ax_bl, ax_br) = axes

# -- Top panels: daily tx count ------------------------------------------------
for ax, (name, addrs, base_color) in zip([ax_tl, ax_tr], [
    ("Wintermute", SEARCHERS["Wintermute"], "#1f77b4"),
    ("Selini",     SEARCHERS["Selini"],     "#ff7f0e"),
]):
    df = txs[txs["tx_to_address"].isin(addrs)].copy()
    df["date"] = pd.to_datetime(df["block_time"].dt.date)
    daily = (
        df.groupby(["date", "timeboosted"]).size()
        .unstack(fill_value=0)
        .rename(columns={True: "TB", False: "Non-TB"})
        .reset_index()
    )
    daily["date"] = pd.to_datetime(daily["date"])
    t_min, t_max = daily["date"].min(), daily["date"].max()

    for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
        x0 = t_min if z_start is None else z_start.tz_convert(None).to_datetime64()
        x1 = t_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
        ax.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)

    ax.bar(daily["date"] - _BAR_OFFSET, daily["TB"],     width=_BAR_W,
           color=base_color, alpha=0.85, label="Timeboosted", zorder=3)
    ax.bar(daily["date"] + _BAR_OFFSET, daily["Non-TB"], width=_BAR_W,
           color=base_color, alpha=0.35, label="Regular", zorder=3)
    _add_event_lines(ax)

    for z_start, z_end, z_label in ZONE_LABELS:
        x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
        x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
        mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
        ax.text(mid, 1.01, z_label, ha="center", va="bottom",
                fontsize=FONT["annot"]-3, color="0.25", style="italic", zorder=6, transform=ax.get_xaxis_transform(), clip_on=False)

    ax.set_title(name, fontsize=FONT["title"])
    ax.set_ylabel("Transaction count")
    ax.set_xlim(t_min - _BAR_W, t_max + _BAR_W)
    ax.set_axisbelow(True)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.legend(fontsize=FONT["legend"])

# -- Bottom-left: daily total PnL ----------------------------------------------
_vol_df = pd.DataFrame({"date": pd.to_datetime(daily_vol["timestamp"]), "daily_vol": daily_vol["daily_vol"].values})
_SHORT = {"Wintermute": "WM", "Selini": "Sel"}
_searcher_data = []
for name, summary, color in [
    ("Wintermute", wm_summary,  "#1f77b4"),
    ("Selini",     sel_summary, "#ff7f0e"),
]:
    df = summary[h].copy()
    df["date"] = pd.to_datetime(df["date"])
    full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    df = df.set_index("date").reindex(full_range).reset_index().rename(columns={"index": "date"})
    _searcher_data.append((name, df, color))

t_min = min(d["date"].min() for _, d, _ in _searcher_data)
t_max = max(d["date"].max() for _, d, _ in _searcher_data)

for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_datetime64()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
    ax_bl.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)

for name, df, color in _searcher_data:
    ax_bl.plot(df["date"], df[f"tb_pnl_t{h}"],  color=color, linewidth=1.8,
               marker="o", markersize=3, label=f"{_SHORT[name]} TB")
    ax_bl.plot(df["date"], df[f"ntb_pnl_t{h}"], color=color, linewidth=1.8,
               marker="s", markersize=3, linestyle="--", alpha=0.6, label=f"{_SHORT[name]} Regular")

ax_bl.axhline(0, color="black", linewidth=0.6, linestyle=":")
_add_event_lines(ax_bl)

ax_bl_vol = _add_vol_axis(ax_bl, _vol_df)

for z_start, z_end, z_label in ZONE_LABELS:
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax_bl.text(mid, 1.01, z_label, ha="center", va="bottom",
               fontsize=FONT["annot"]-3, color="0.25", style="italic", zorder=6, transform=ax_bl.get_xaxis_transform(), clip_on=False)

ax_bl.set_ylabel("Daily PnL (USD)")
ax_bl.set_xlim(t_min, t_max)
ax_bl.grid(axis="y", linestyle="--", alpha=0.4)
ax_bl.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax_bl.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.setp(ax_bl.get_xticklabels(), rotation=30, ha="right")
_l1, _lb1 = ax_bl.get_legend_handles_labels()
_l2, _lb2 = ax_bl_vol.get_legend_handles_labels()
ax_bl.legend(_l1 + _l2, _lb1 + _lb2, fontsize=FONT["legend"]-1, loc="upper right", bbox_to_anchor=(1, 0.92))

# -- Bottom-right: daily avg PnL per trade -------------------------------------
_avg_data = []
for name, addrs, color in [
    ("Wintermute", SEARCHERS["Wintermute"], "#1f77b4"),
    ("Selini",     SEARCHERS["Selini"],     "#ff7f0e"),
]:
    df = txs[txs["tx_to_address"].isin(addrs)].copy()
    df["date"] = pd.to_datetime(df["block_time"]).dt.date
    avg = (
        df[df[pnl_col] > 0]
        .groupby(["date", "timeboosted"])[pnl_col]
        .mean()
        .unstack(fill_value=np.nan)
        .rename(columns={True: "tb_avg", False: "ntb_avg"})
        .reset_index()
    )
    avg["date"] = pd.to_datetime(avg["date"])
    full_range = pd.date_range(avg["date"].min(), avg["date"].max(), freq="D")
    avg = avg.set_index("date").reindex(full_range).reset_index().rename(columns={"index": "date"})
    _avg_data.append((name, avg, color))

t_min = min(d["date"].min() for _, d, _ in _avg_data)
t_max = max(d["date"].max() for _, d, _ in _avg_data)

for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_datetime64()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
    ax_br.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)

for name, avg, color in _avg_data:
    ax_br.plot(avg["date"], avg["tb_avg"],  color=color, linewidth=1.8,
               marker="o", markersize=3, label=f"{_SHORT[name]} TB")
    ax_br.plot(avg["date"], avg["ntb_avg"], color=color, linewidth=1.8,
               marker="s", markersize=3, linestyle="--", alpha=0.6, label=f"{_SHORT[name]} Regular")

ax_br.axhline(0, color="black", linewidth=0.6, linestyle=":")
_add_event_lines(ax_br)


for z_start, z_end, z_label in ZONE_LABELS:
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax_br.text(mid, 1.01, z_label, ha="center", va="bottom",
               fontsize=FONT["annot"]-3, color="0.25", style="italic", zorder=6, transform=ax_br.get_xaxis_transform(), clip_on=False)

ax_br.set_ylabel("Avg PnL per trade (USD)")
ax_br.set_xlim(t_min, t_max)
ax_br.grid(axis="y", linestyle="--", alpha=0.4)
ax_br.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax_br.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.setp(ax_br.get_xticklabels(), rotation=30, ha="right")
ax_br.legend(fontsize=FONT["legend"]-1, loc="upper right", bbox_to_anchor=(1, 0.82))

fig.suptitle(f"Daily CEX-DEX Transaction Activity and PnL by Searcher at {h}s Markout", fontsize=FONT["title"])
fig.tight_layout()
plt.savefig("figures/daily_activity_pnl_4panel.pdf", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
# Same 4-panel figure but top panels count only positive-PnL transactions
_BAR_W      = pd.Timedelta(hours=9.6)
_BAR_OFFSET = pd.Timedelta(hours=4.8)
h = 5
pnl_col = f"pnl_t{h}"

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
(ax_tl, ax_tr), (ax_bl, ax_br) = axes

# -- Top panels: daily tx count (positive PnL only) ----------------------------
for ax, (name, addrs, base_color) in zip([ax_tl, ax_tr], [
    ("Wintermute", SEARCHERS["Wintermute"], "#1f77b4"),
    ("Selini",     SEARCHERS["Selini"],     "#ff7f0e"),
]):
    df = txs[txs["tx_to_address"].isin(addrs)].copy()
    df = df[df[pnl_col] > 0]                          # positive PnL only
    df["date"] = pd.to_datetime(df["block_time"].dt.date)
    daily = (
        df.groupby(["date", "timeboosted"]).size()
        .unstack(fill_value=0)
        .rename(columns={True: "TB", False: "Non-TB"})
        .reset_index()
    )
    daily["date"] = pd.to_datetime(daily["date"])
    t_min, t_max = daily["date"].min(), daily["date"].max()

    for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
        x0 = t_min if z_start is None else z_start.tz_convert(None).to_datetime64()
        x1 = t_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
        ax.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)

    ax.bar(daily["date"] - _BAR_OFFSET, daily["TB"],     width=_BAR_W,
           color=base_color, alpha=0.85, label="Timeboosted", zorder=3)
    ax.bar(daily["date"] + _BAR_OFFSET, daily["Non-TB"], width=_BAR_W,
           color=base_color, alpha=0.35, label="Regular", zorder=3)
    _add_event_lines(ax)

    for z_start, z_end, z_label in ZONE_LABELS:
        x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
        x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
        mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
        ax.text(mid, 1.01, z_label, ha="center", va="bottom",
                fontsize=FONT["annot"]-3, color="0.25", style="italic", zorder=6, transform=ax.get_xaxis_transform(), clip_on=False)

    ax.set_title(name, fontsize=FONT["title"], pad=30)
    ax.set_ylabel("Transaction count")
    ax.set_xlim(t_min - _BAR_W, t_max + _BAR_W)
    ax.set_axisbelow(True)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.legend(fontsize=FONT["legend"])

# -- Bottom-left: daily total PnL ----------------------------------------------
_vol_df = pd.DataFrame({"date": pd.to_datetime(daily_vol["timestamp"]), "daily_vol": daily_vol["daily_vol"].values})
_SHORT = {"Wintermute": "WM", "Selini": "Sel"}
_searcher_data = []
for name, summary, color in [
    ("Wintermute", wm_summary,  "#1f77b4"),
    ("Selini",     sel_summary, "#ff7f0e"),
]:
    df = summary[h].copy()
    df["date"] = pd.to_datetime(df["date"])
    full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    df = df.set_index("date").reindex(full_range).reset_index().rename(columns={"index": "date"})
    _searcher_data.append((name, df, color))

t_min = min(d["date"].min() for _, d, _ in _searcher_data)
t_max = max(d["date"].max() for _, d, _ in _searcher_data)

for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_datetime64()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
    ax_bl.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)

for name, df, color in _searcher_data:
    ax_bl.plot(df["date"], df[f"tb_pnl_t{h}"],  color=color, linewidth=1.8,
               marker="o", markersize=3, label=f"{_SHORT[name]} TB")
    ax_bl.plot(df["date"], df[f"ntb_pnl_t{h}"], color=color, linewidth=1.8,
               marker="s", markersize=3, linestyle="--", alpha=0.6, label=f"{_SHORT[name]} Regular")

ax_bl.axhline(0, color="black", linewidth=0.6, linestyle=":")
_add_event_lines(ax_bl)

ax_bl_vol = _add_vol_axis(ax_bl, _vol_df)

for z_start, z_end, z_label in ZONE_LABELS:
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax_bl.text(mid, 1.01, z_label, ha="center", va="bottom",
               fontsize=FONT["annot"]-3, color="0.25", style="italic", zorder=6, transform=ax_bl.get_xaxis_transform(), clip_on=False)

ax_bl.set_ylabel("Daily PnL (USD)")
ax_bl.set_xlim(t_min, t_max)
ax_bl.grid(axis="y", linestyle="--", alpha=0.4)
ax_bl.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax_bl.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.setp(ax_bl.get_xticklabels(), rotation=30, ha="right")
_l1, _lb1 = ax_bl.get_legend_handles_labels()
_l2, _lb2 = ax_bl_vol.get_legend_handles_labels()
ax_bl.legend(_l1 + _l2, _lb1 + _lb2, fontsize=FONT["legend"]-2, loc="upper right")

# -- Bottom-right: daily avg PnL per trade -------------------------------------
_avg_data = []
for name, addrs, color in [
    ("Wintermute", SEARCHERS["Wintermute"], "#1f77b4"),
    ("Selini",     SEARCHERS["Selini"],     "#ff7f0e"),
]:
    df = txs[txs["tx_to_address"].isin(addrs)].copy()
    df["date"] = pd.to_datetime(df["block_time"]).dt.date
    avg = (
        df[df[pnl_col] > 0]
        .groupby(["date", "timeboosted"])[pnl_col]
        .mean()
        .unstack(fill_value=np.nan)
        .rename(columns={True: "tb_avg", False: "ntb_avg"})
        .reset_index()
    )
    avg["date"] = pd.to_datetime(avg["date"])
    full_range = pd.date_range(avg["date"].min(), avg["date"].max(), freq="D")
    avg = avg.set_index("date").reindex(full_range).reset_index().rename(columns={"index": "date"})
    _avg_data.append((name, avg, color))

t_min = min(d["date"].min() for _, d, _ in _avg_data)
t_max = max(d["date"].max() for _, d, _ in _avg_data)

for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_datetime64()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
    ax_br.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)

for name, avg, color in _avg_data:
    ax_br.plot(avg["date"], avg["tb_avg"],  color=color, linewidth=1.8,
               marker="o", markersize=3, label=f"{_SHORT[name]} TB")
    ax_br.plot(avg["date"], avg["ntb_avg"], color=color, linewidth=1.8,
               marker="s", markersize=3, linestyle="--", alpha=0.6, label=f"{_SHORT[name]} Regular")

ax_br.axhline(0, color="black", linewidth=0.6, linestyle=":")
_add_event_lines(ax_br)


for z_start, z_end, z_label in ZONE_LABELS:
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax_br.text(mid, 1.01, z_label, ha="center", va="bottom",
               fontsize=FONT["annot"]-3, color="0.25", style="italic", zorder=6, transform=ax_br.get_xaxis_transform(), clip_on=False)

ax_br.set_ylabel("Avg PnL per trade (USD)")
ax_br.set_xlim(t_min, t_max)
ax_br.grid(axis="y", linestyle="--", alpha=0.4)
ax_br.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax_br.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.setp(ax_br.get_xticklabels(), rotation=30, ha="right")
ax_br.legend(fontsize=FONT["legend"]-1, loc="upper right")

fig.suptitle(f"Daily CEX-DEX Activity and PnL by Searcher at {h}s Markout (Positive PnL Trades Only)", fontsize=FONT["title"])
fig.tight_layout()
plt.savefig("figures/daily_activity_pnl_4panel_pos.pdf", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
# Timeboosted positive-PnL tx count over time: by searcher, colored by auction winner
h = 5
pnl_col = f"pnl_t{h}"

# Build winner lookup: auction_round -> winner_name (collapse to WM / Sel / Kairos / Other)
_winner_map = (
    auctions[["auction_round", "winner_name"]]
    .dropna(subset=["auction_round"])
    .assign(auction_round=lambda d: d["auction_round"].astype(int))
    .set_index("auction_round")["winner_name"]
    .map(lambda w: w if w in ("Wintermute", "Selini", "Kairos") else "Other")
)

WINNER_ORDER  = ["Wintermute", "Selini", "Kairos", "Other"]
WINNER_COLORS_TB = {
    "Wintermute": "#1f77b4",
    "Selini":     "#ff7f0e",
    "Kairos":     "#9467bd",
    "Other":      "#7f7f7f",
}

_BAR_W = pd.Timedelta(hours=19.2)   # full-day bar (single stack)

fig, (ax_wm, ax_sel) = plt.subplots(2, 1, figsize=FIG["wide_tall"], sharex=True)

for ax, (name, addrs) in zip([ax_wm, ax_sel], [
    ("Wintermute", SEARCHERS["Wintermute"]),
    ("Selini",     SEARCHERS["Selini"]),
]):
    df = txs[
        txs["tx_to_address"].isin(addrs) &
        (txs["timeboosted"] == True) &
        (txs[pnl_col] > 0)
    ].copy()
    df["date"] = pd.to_datetime(df["block_time"].dt.date)
    df["auction_round_int"] = df["auction_round"].astype("Int64")
    df["winner"] = df["auction_round_int"].map(_winner_map).fillna("Other")

    daily = (
        df.groupby(["date", "winner"]).size()
        .unstack(fill_value=0)
        .reindex(columns=WINNER_ORDER, fill_value=0)
    )
    full_idx = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
    daily = daily.reindex(full_idx, fill_value=0)
    daily.index = pd.to_datetime(daily.index)

    t_min, t_max = daily.index.min(), daily.index.max()

    for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
        x0 = t_min if z_start is None else z_start.tz_convert(None).to_datetime64()
        x1 = t_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
        ax.axvspan(x0, x1, alpha=0.12, color=ZONE_COLORS[i], zorder=0)

    bottom = np.zeros(len(daily))
    for winner in WINNER_ORDER:
        if winner not in daily.columns:
            continue
        vals = daily[winner].values
        ax.bar(daily.index, vals, bottom=bottom, width=_BAR_W,
               color=WINNER_COLORS_TB[winner], alpha=0.85,
               label=winner, zorder=3)
        bottom += vals

    _add_event_lines(ax)

    for z_start, z_end, z_label in ZONE_LABELS:
        x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
        x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
        mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
        ax.text(mid, 1.01, z_label, ha="center", va="bottom",
                fontsize=FONT["annot"], color="0.25", style="italic",
                zorder=6, transform=ax.get_xaxis_transform(), clip_on=False)

    ax.set_title(name, fontsize=FONT["title"], pad=18)
    ax.set_ylabel("TB pos-PnL tx count", fontsize=FONT["label"])
    ax.set_xlim(t_min - _BAR_W, t_max + _BAR_W)
    ax.set_axisbelow(True)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.legend(title="Auction winner", fontsize=FONT["legend"], title_fontsize=FONT["legend"])
    ax.tick_params(labelsize=FONT["tick"])

ax_sel.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax_sel.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.setp(ax_sel.get_xticklabels(), rotation=30, ha="right")

fig.suptitle(f"Daily Timeboosted Positive-PnL Txs by Auction Winner ({h}s Markout)", fontsize=FONT["title"])
fig.tight_layout()
plt.savefig("figures/tb_pospnl_by_winner.pdf", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
# Summary: TB positive-PnL tx counts and PnL by searcher × auction winner
h = 5
pnl_col = f"pnl_t{h}"

df_all = txs[
    (txs["timeboosted"] == True) &
    (txs[pnl_col] > 0)
].copy()
df_all["auction_round_int"] = df_all["auction_round"].astype("Int64")
df_all["winner"] = df_all["auction_round_int"].map(_winner_map).fillna("Other")

print(f"{'Searcher':<14} {'Auction Winner':<14} {'TX Count':>10} {'Total PnL':>14}")
print("-" * 56)
for sname, addrs in SEARCHERS.items():
    sub = df_all[df_all["tx_to_address"].isin(addrs)]
    for winner in WINNER_ORDER:
        w_sub = sub[sub["winner"] == winner]
        if len(w_sub) == 0:
            continue
        print(f"{sname:<14} {winner:<14} {len(w_sub):>10,} ${w_sub[pnl_col].sum():>13,.0f}")
    total = sub[pnl_col].sum()
    print(f"{sname:<14} {'TOTAL':<14} {len(sub):>10,} ${total:>13,.0f}")
    print()


In [ ]:
# Total timeboosted tx counts by searcher (all PnL directions)
tb = txs[txs["timeboosted"] == False]
for name, addrs in SEARCHERS.items():
    n = tb["tx_to_address"].isin(addrs).sum()
    print(f"{name:<14} {n:>10,}")
print(f"{'Total':<14} {tb['tx_to_address'].isin(set().union(*SEARCHERS.values())).sum():>10,}")


In [ ]:
# Round discrepancy: bids_all vs auctions
import glob, pandas as pd

bid_files = sorted(glob.glob("data/case_study/bids/bids_*.csv"))
_bids = pd.concat([pd.read_csv(f) for f in bid_files], ignore_index=True)

bids_rounds    = set(_bids["Round"].dropna().astype(int))
auction_rounds = set(auctions["auction_round"].dropna().astype(int))

in_bids_not_auctions = sorted(bids_rounds - auction_rounds)
in_auctions_not_bids = sorted(auction_rounds - bids_rounds)

print(f"Unique rounds in bids    : {len(bids_rounds):,}")
print(f"Unique rounds in auctions: {len(auction_rounds):,}")
print()
print(f"In bids but NOT in auctions ({len(in_bids_not_auctions)}): {in_bids_not_auctions[:20]}")
print()
print(f"In auctions but NOT in bids ({len(in_auctions_not_bids)}): {in_auctions_not_bids[:20]}")

if in_auctions_not_bids:
    sub = auctions[auctions["auction_round"].isin(in_auctions_not_bids)].copy()
    _rst = pd.to_datetime(sub["round_start_time"])
    if _rst.dt.tz is not None:
        _rst = _rst.dt.tz_convert(None)
    sub["date"] = _rst.dt.date
    print(f"\nDate range of auction-only rounds: {sub['date'].min()} -> {sub['date'].max()}")
    print(sub["date"].value_counts().sort_index().to_string())


In [ ]:
# Average daily TB / Regular tx count per searcher per zone
zones = [lbl for _, _, lbl in ZONE_LABELS]

for name, addrs in SEARCHERS.items():
    df = txs[txs["tx_to_address"].isin(addrs)].copy()
    df["date"] = pd.to_datetime(df["block_time"].dt.date)
    df["zone"] = df["date"].apply(lambda d: _assign_zone(d.date()))

    daily = (
        df.groupby(["date", "zone", "timeboosted"]).size()
        .unstack(fill_value=0)
        .rename(columns={True: "TB", False: "Regular"})
        .reset_index()
    )

    print(f"{name}")
    print(f"  {'Zone':<30} {'Avg TB/day':>12} {'Avg Regular/day':>16}")
    print(f"  {'-'*60}")
    for z in zones:
        sub = daily[daily["zone"] == z]
        avg_tb  = sub["TB"].mean()      if "TB"      in sub.columns else float("nan")
        avg_reg = sub["Regular"].mean() if "Regular" in sub.columns else float("nan")
        print(f"  {z.replace(chr(10), ' '):<30} {avg_tb:>12.1f} {avg_reg:>16.1f}")
    print()


In [ ]:
from scipy.stats import pearsonr

# Correlation: daily PnL (TB / Non-TB / Total) vs ETH volatility, per searcher per zone
h = 5

# Build a date-indexed vol Series for clean mapping
_vol_idx = pd.Series(
    daily_vol["daily_vol"].values,
    index=pd.to_datetime(daily_vol["timestamp"])
)

_searchers = [
    ("Wintermute", wm_summary),
    ("Selini",     sel_summary),
]
_zones   = [lbl for _, _, lbl in ZONE_LABELS]
_metrics = [("TB PnL",     f"tb_pnl_t{h}"),
            ("Non-TB PnL", f"ntb_pnl_t{h}"),
            ("Total PnL",  "total_pnl")]

def _corr_stars(r, p):
    s = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    return f"{r:+.3f}", f"{p:.3f}{s}"

for searcher_name, summary in _searchers:
    df = summary[h].copy()
    df["date"] = pd.to_datetime(df["date"])
    df["total_pnl"] = df[f"tb_pnl_t{h}"] + df[f"ntb_pnl_t{h}"]
    df["daily_vol"] = df["date"].map(_vol_idx)
    df = df.dropna(subset=["daily_vol"])
    df["zone"] = df["date"].apply(lambda d: _assign_zone(d.date()))

    print(f"{searcher_name}")
    print(f"  {'Metric':<14} {'Zone':<30} {'r':>8} {'p':>12} {'n':>5}")
    print(f"  {'-'*65}")

    for metric_label, col in _metrics:
        first = True
        for zone in _zones:
            sub = df[df["zone"] == zone][["daily_vol", col]].dropna()
            n = len(sub)
            if n < 3:
                r_str, p_str = "n/a", "n/a"
            else:
                r, p = pearsonr(sub["daily_vol"], sub[col])
                r_str, p_str = _corr_stars(r, p)
            label = metric_label if first else ""
            print(f"  {label:<14} {zone.replace(chr(10), ' '):<30} {r_str:>8} {p_str:>12} {n:>5}")
            first = False
        print()
    print()


## Reverts & Success Rates
Load revert data, compute success/failure rates, and compare CEX-DEX shares across timeboosted vs non-timeboosted transactions.

In [ ]:
csv_files = glob.glob(f"data/case_study/reverts/*_parsed.csv")
reverts = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

reverts["block_time"] = pd.to_datetime(reverts["block_time"], utc=True)
auctions["round_start_time"] = pd.to_datetime(auctions["round_start_time"], utc=True)
auctions["round_end_time"]   = pd.to_datetime(auctions["round_end_time"], utc=True)

# Remove timezone for merge_asof compatibility
reverts["block_time"]        = reverts["block_time"].dt.tz_convert(None)
auctions["round_start_time"] = auctions["round_start_time"].dt.tz_convert(None)
auctions["round_end_time"]   = auctions["round_end_time"].dt.tz_convert(None)

reverts = reverts.sort_values("block_time")
auctions = auctions.sort_values("round_start_time")

reverts["minute_start"] = reverts["block_time"].dt.floor("min")
reverts["time_since_minute_start"] = (
    reverts["block_time"] - reverts["minute_start"]
).dt.total_seconds()

reverts["timeboosted"] = reverts["timeboosted"].fillna(False)

In [ ]:
# Print total transactions and reverts
total_txs = len(reverts)
total_reverts = reverts["success"].sum()

print(f"Total transactions: {total_txs}")
print(f"Total reverts: {total_reverts} ({100 * total_reverts / total_txs:.2f}%)")

In [ ]:
all_wm_txs = reverts[reverts["tx_to_address"].isin(SEARCHERS["Wintermute"])].copy()
all_sel_txs = reverts[reverts["tx_to_address"].isin(SEARCHERS["Selini"])].copy()

wm_success_not_cex_dex = all_wm_txs[(all_wm_txs["success"] == 1) & (~all_wm_txs["tx_hash"].isin(wm_txs["tx_hash"]))]
sel_success_not_cex_dex = all_sel_txs[(all_sel_txs["success"] == 1) & (~all_sel_txs["tx_hash"].isin(sel_txs["tx_hash"]))]

wm_tb = all_wm_txs[all_wm_txs["timeboosted"] == True]
wm_non_tb = all_wm_txs[all_wm_txs["timeboosted"] == False]
sel_tb = all_sel_txs[all_sel_txs["timeboosted"] == True]
sel_non_tb = all_sel_txs[all_sel_txs["timeboosted"] == False]

# Boolean flags for filtering
for df, cexdex_txs in [(all_wm_txs, wm_txs), (all_sel_txs, sel_txs)]:
    df["is_tb"]          = df["timeboosted"] == True
    df["is_ntb"]         = df["timeboosted"] == False
    df["is_success"]     = df["success"] == 1
    df["is_revert"]      = ~df["is_success"]
    df["is_cexdex"]      = df["tx_hash"].isin(cexdex_txs["tx_hash"])
    df["is_non_cexdex"]  = df["is_success"] & (~df["is_cexdex"])

summary = {}
for name, all_txs, cexdex_txs, tb, ntb in [
    ("Wintermute", all_wm_txs, wm_txs, wm_tb, wm_non_tb),
    ("Selini",     all_sel_txs, sel_txs, sel_tb, sel_non_tb),
]:
    summary[name] = {
        "cexdex":     len(cexdex_txs),
        "non_cexdex": len(all_txs[(all_txs["success"] == 1) & (~all_txs["tx_hash"].isin(cexdex_txs["tx_hash"]))]),
        "tb_total":   len(tb),
        "tb_rate":    tb["success"].mean(),
        "ntb_total":  len(ntb),
        "ntb_rate":   ntb["success"].mean(),
        "tb_cexdex":  len(tb[tb["tx_hash"].isin(cexdex_txs["tx_hash"])]),
        "ntb_cexdex": len(ntb[ntb["tx_hash"].isin(cexdex_txs["tx_hash"])]),
    }

In [ ]:
print(reverts['tx_to_address'].value_counts().head(20))

In [ ]:
searcher_colors = {"Wintermute": "#1f77b4", "Selini": "#ff7f0e"}

daily_parts = []
for name, df in {"Wintermute": all_wm_txs, "Selini": all_sel_txs}.items():
    d = df.copy()
    d["date"] = pd.to_datetime(d["block_time"]).dt.normalize()
    g = d.groupby("date").agg(total=("success", "count"), success=("success", "sum")).reset_index()
    g["reverts"]  = g["total"] - g["success"]
    g["searcher"] = name
    daily_parts.append(g)

daily_sr = pd.concat(daily_parts, ignore_index=True)

date_min = daily_sr["date"].min()
date_max = daily_sr["date"].max()

fig, ax = plt.subplots(figsize=(14, 5))

for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_datetime64()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
    ax.axvspan(x0, x1, alpha=0.12, color=ZONE_COLORS[i], zorder=0)
_add_event_lines(ax)

for name, grp in daily_sr.groupby("searcher"):
    grp = grp.sort_values("date")
    c = searcher_colors[name]
    ax.plot(grp["date"], grp["success"], color=c, linewidth=1.6,
            marker="o", markersize=2.5, linestyle="-",  label=f"{name} Success")
    ax.plot(grp["date"], grp["reverts"], color=c, linewidth=1.6,
            marker="o", markersize=2.5, linestyle="--", label=f"{name} Reverts", alpha=0.7)

for z_start, z_end, z_label in ZONE_LABELS:
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax.text(mid, 1.01, z_label, ha="center", va="bottom",
            fontsize=7.5, color="0.25", style="italic", zorder=6, transform=ax.get_xaxis_transform(), clip_on=False)

ax.set_ylabel("Daily Tx Count")
ax.legend(fontsize=FONT["legend"])
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.suptitle("Daily Success & Revert Counts by Searcher", fontsize=FONT["title"])
fig.tight_layout()
plt.savefig("figures/success_revert_over_time.pdf", bbox_inches="tight", dpi=300)
plt.show()